# CASMI26 · Generated Class-3 candidates

### Submission pipeline and run notes

This notebook builds a spectral-retrieval candidate set, augments it with one-step non-parental (NP) transformations, fits two rankers, and writes the competition submission. The generated-candidate run and its no-generation comparison are both saved.

> **Run snapshot**  
> 400 test molecules · 1,213 spectra · 2.54M library spectra · 712,199 candidate structures  
> Generated candidates ranked first for **0.8%** of test molecules and made up **11.2%** of the top-25 list on average. At least one generated candidate was available for **44.8%** of molecules.

These are coverage and ranking diagnostics from the saved run, not a measured leaderboard lift. The notebook also writes a same-pipeline baseline so the two submissions can be compared downstream.


## 1 · Reproducible Kaggle setup

The first code cell embeds the engine modules and wires the competition and model assets into `/kaggle/working/casmi`. The implementation is kept inline so the notebook can run as a self-contained Kaggle artifact. Its source is intentionally collapsed; expand it when you need to inspect or edit the engine.


In [ ]:
SRC = {"pv.py": "\"\"\"prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied\nverbatim where possible so local simulations and the Kaggle notebook share one source of truth.\nAdditions are marked `# ADDED`.\n\"\"\"\nimport math\nimport numpy as np\nfrom numba import njit, prange\n\n\nclass CFG:\n    PPM_WIN = 10.0\n    PPM_FALLBACK = 30.0\n    INT_FLOOR = 0.002\n    MAX_PEAKS = 256\n    MZ_TOL = 0.01\n    INT_POWER = 1.0\n    ENT_WEIGHT = True\n    ANALOG_WIN = 200.0\n    N_ANALOG = 80\n    SIM_POWER = 3.0\n    W1_PRIORS = (0.30, 0.60)\n    SEEDS = (0, 1, 2, 3)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n    TOPN = 25\n\n\n# ----------------------------------------------------------------------------------------------\n# similarity kernels\n# ----------------------------------------------------------------------------------------------\n@njit(cache=True, fastmath=True)\ndef _clean(mz, it, floor, topk, power, ent_weight):\n    n = len(mz)\n    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx: mx = it[i]\n    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx; c = 0\n    for i in range(n):\n        if it[i] >= thr: c += 1\n    idx = np.empty(c, np.int64); j = 0\n    for i in range(n):\n        if it[i] >= thr: idx[j] = i; j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c): v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk:]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk): k2[i] = idx[o[i]]\n        k2.sort(); idx = k2; c = topk\n    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v\n    if s > 0:\n        for i in range(c): oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0: S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S; s2 = 0.0\n            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]\n            if s2 > 0:\n                for i in range(c): oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0; j = 0; n = len(qmz); m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0; tot = 0.0\n    buf = np.empty(n + m, np.float64); b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol: buf[b] = qp[i]; i += 1; b += 1\n        elif d > tol: buf[b] = cp[j]; j += 1; b += 1\n        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1\n    while i < n: buf[b] = qp[i]; i += 1; b += 1\n    while j < m: buf[b] = cp[j]; j += 1; b += 1\n    for x in range(b): tot += buf[x]\n    if tot <= 0: return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0: SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\n    return out\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):\n    a = entropy_sim(qmz, qp, cmz, cp, tol)\n    if shift > -0.001 and shift < 0.001: return a\n    sm = np.empty(len(cmz), np.float32)\n    for i in range(len(cmz)): sm[i] = cmz[i] + shift\n    b = entropy_sim(qmz, qp, sm, cp, tol)\n    return a if a > b else b\n\n\n# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):\n    out = np.zeros(hi - lo, np.float32)\n    for k in prange(hi - lo):\n        a = roff[lo + k]; b = roff[lo + k + 1]\n        if b <= a: continue\n        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])\n    return out\n\n\n# ADDED: library search where each reference is shifted by (query precursor - reference precursor):\n# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])\n    return out\n\n\n# ADDED: clean a list of spectra into a flat store\n@njit(cache=True)\ndef clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):\n    n = len(rows)\n    lens = np.zeros(n + 1, np.int64)\n    tmp_m = []\n    tmp_i = []\n    for k in range(n):\n        r = rows[k]\n        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)\n        tmp_m.append(cm); tmp_i.append(cp)\n        lens[k + 1] = lens[k] + len(cm)\n    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)\n    for k in range(n):\n        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]\n    return lens, om, oi\n\n\n# ----------------------------------------------------------------------------------------------\n# adducts\n# ----------------------------------------------------------------------------------------------\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\nE = 0.00054857990; PROTON = MASS['H'] - E; H2O = 2 * MASS['H'] + MASS['O']\nNH4 = MASS['N'] + 4 * MASS['H']; FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']\nACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']\nADDUCTS = {\n    \"[M+H]+\": (1, 1, PROTON), \"[M+NH4]+\": (1, 1, NH4 - E), \"[M+Na]+\": (1, 1, MASS['Na'] - E),\n    \"[M+K]+\": (1, 1, MASS['K'] - E), \"[M-H2O+H]+\": (1, 1, PROTON - H2O), \"[M-2H2O+H]+\": (1, 1, PROTON - 2 * H2O),\n    \"[M+2H]2+\": (1, 2, 2 * PROTON), \"[M]+\": (1, 1, -E), \"[M-H2O]+\": (1, 1, -E - H2O),\n    \"[M+CH3OH+H]+\": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),\n    \"[M+CH3CN+H]+\": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),\n    \"[M-H]-\": (1, 1, -PROTON), \"[M-H2O-H]-\": (1, 1, -PROTON - H2O), \"[M+CH2O2-H]-\": (1, 1, FORMATE - PROTON),\n    \"[M+C2H4O2-H]-\": (1, 1, ACETATE - PROTON), \"[M+Cl]-\": (1, 1, MASS['Cl'] + E), \"[M]-\": (1, 1, E),\n    \"[M-2H]-\": (1, 2, -2 * PROTON), \"[M+Na-2H]-\": (1, 1, MASS['Na'] - 2 * PROTON),\n    \"[2M+H]+\": (2, 1, PROTON), \"[2M+Na]+\": (2, 1, MASS['Na'] - E), \"[2M+NH4]+\": (2, 1, NH4 - E),\n    \"[2M+K]+\": (2, 1, MASS['K'] - E), \"[2M-H]-\": (2, 1, -PROTON), \"[2M+CH2O2-H]-\": (2, 1, FORMATE - PROTON),\n    \"[2M+C2H4O2-H]-\": (2, 1, ACETATE - PROTON), \"[2M+Na-2H]-\": (2, 1, MASS['Na'] - 2 * PROTON),\n    \"[3M+H]+\": (3, 1, PROTON), \"[3M-H]-\": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz, adduct):\n    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = (ad == a)\n        if m.any(): out[m] = (mz[m] * z - d) / n\n    return out\n\n\n# ----------------------------------------------------------------------------------------------\n# ranking features (31)\n# ----------------------------------------------------------------------------------------------\nN_ANALOG = 80\nP_SIM = 3.0\nN_FEAT = 31\n\n\ndef _rank_norm(x):\n    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)\n\n\ndef _z(x):\n    s = x.std()\n    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)\n\n\ndef rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):\n    nc = cand_fp.shape[0]\n    cf = cand_fp.astype(np.float32); cs = cf.sum(1)\n    lv = np.asarray(cand_lib, np.float32)\n    lvmax = float(lv.max()) if nc else 0.0\n    if analog_fp is not None and len(analog_sim):\n        af = analog_fp.astype(np.float32); asum = af.sum(1)\n        inter = cf @ af.T\n        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)\n        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)\n        ap = (tan * (w ** P_SIM)[None, :]).max(1)\n        a1 = (tan * w[None, :]).max(1)\n        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])\n        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)\n    else:\n        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0\n    apmax = float(ap.max()) if nc else 0.0\n    if model_logits is not None:\n        raw = cf @ np.asarray(model_logits, np.float32)\n        nrm = raw / np.sqrt(np.maximum(cs, 1.0))\n        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),\n                 (raw == raw.max()).astype(np.float32)]\n    else:\n        mfeat = [np.zeros(nc, np.float32)] * 6\n    if model_logits is not None and nc:\n        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))\n        lbest = int(np.argmax(lv)) if lvmax > 0 else -1\n        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0\n        abest = int(np.argmax(ap)) if apmax > 0 else -1\n        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0\n        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),\n                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]\n    else:\n        xfeat = [np.zeros(nc, np.float32)] * 6\n    if frag is not None:\n        fr = np.asarray(frag, np.float32)\n        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]\n    else:\n        ffeat = [np.zeros(nc, np.float32)] * 4\n    return np.column_stack([\n        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),\n        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,\n        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),\n        np.full(nc, np.log(max(nc, 1))),\n        *mfeat, *ffeat, *xfeat,\n    ]).astype(np.float32)\n\n\n# ----------------------------------------------------------------------------------------------\n# MetFrag-lite\n# ----------------------------------------------------------------------------------------------\nAMU = {'C': 12.0, 'H': 1.00782503207, 'N': 14.0030740048, 'O': 15.9949146196, 'P': 30.97376163,\n       'S': 31.97207100, 'F': 18.99840322, 'Cl': 34.96885268, 'Br': 78.9183371, 'I': 126.904473,\n       'Na': 22.9897692809, 'K': 38.96370668, 'Si': 27.9769265325, 'B': 11.0093054, 'Se': 79.9165213}\nH = AMU['H']\n\n\ndef mol_graph(smi):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog('rdApp.*')\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H\n    if (w == 0).any(): return None\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\n    return w, bonds, n\n\n\ndef _components(n, bonds, drop):\n    adj = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop: continue\n        adj[a].append(b); adj[b].append(a)\n    seen = np.zeros(n, bool); comps = []\n    for s in range(n):\n        if seen[s]: continue\n        stack = [s]; seen[s] = True; cur = [s]\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi, max_breaks=2, max_bonds=34):\n    g = mol_graph(smi)\n    if g is None: return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds: return np.array([w.sum()])\n    out = {w.sum()}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef frag_masses_safe(smi):\n    try: return fragment_masses(smi)\n    except Exception: return np.zeros(0)\n\n\n# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)\nCARRIERS = {\n    \"[M+H]+\": (PROTON,), \"[M-H2O+H]+\": (PROTON,), \"[M-2H2O+H]+\": (PROTON,),\n    \"[M+NH4]+\": (PROTON, NH4 - E), \"[M+Na]+\": (PROTON, MASS['Na'] - E), \"[M+K]+\": (PROTON, MASS['K'] - E),\n    \"[M-H]-\": (-PROTON,), \"[M-H2O-H]-\": (-PROTON,), \"[M+CH2O2-H]-\": (-PROTON, FORMATE - PROTON),\n    \"[M+Cl]-\": (-PROTON, MASS['Cl'] + E),\n}\n\n\ndef explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    car = CARRIERS.get(adduct, (PROTON,) if \"+\" in adduct[-2:] else (-PROTON,))\n    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n\n\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    ion = []\n    for dh in h_shifts:\n        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))\n    ion = np.sort(np.concatenate(ion))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)", "pv_fp.py": "\"\"\"Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch.\"\"\"\nimport math\nimport numpy as np\n# ----------------------------------------------------------------------------------------------\n# spectrum -> fingerprint model\n# ----------------------------------------------------------------------------------------------\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nFP_MAX_PEAKS = 128\nADDUCT_LIST = [\"[M+H]+\", \"[M+NH4]+\", \"[M+Na]+\", \"[M+K]+\", \"[M-H2O+H]+\", \"[M-2H2O+H]+\", \"[M]+\",\n               \"[M-H]-\", \"[M-H2O-H]-\", \"[M+CH2O2-H]-\", \"[M+C2H4O2-H]-\", \"[M+Cl]-\", \"[M]-\",\n               \"[M+2H]2+\", \"[M-2H]-\", \"[2M+H]+\", \"[2M+Na]+\", \"[2M+NH4]+\", \"[2M-H]-\", \"[2M+K]+\",\n               \"[2M+CH2O2-H]-\", \"[2M+C2H4O2-H]-\", \"[2M+Na-2H]-\", \"[M+Na-2H]-\", \"[M-H2O]+\", \"<unk>\"]\nADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}\nINSTR_LIST = [\"timsTOF\", \"Orbitrap\", \"QTOF\", \"IT\", \"other\"]\n\n\ndef instr_family(s):\n    if s is None: return 4\n    t = str(s).lower()\n    if 'timstof' in t: return 0\n    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1\n    if 'tof' in t: return 2\n    if 'trap' in t or 'qq' in t: return 3\n    return 4\n\n\ndef prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = (mz <= prec_mz + 1.5)\n    mz, it = mz[keep], it[keep]\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    mx = it.max()\n    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = it >= floor * mx\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        order = np.argsort(-it)\n        bucket = (mz // win).astype(np.int64)\n        cnt = {}; sel = []\n        for i in order:\n            b = bucket[i]; c = cnt.get(b, 0)\n            if c < per_win: cnt[b] = c + 1; sel.append(i)\n        sel = np.array(sel)\n        if len(sel) > max_peaks:\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\n        elif len(sel) < max_peaks:\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\n            need = max_peaks - len(sel)\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    v = np.sqrt(it / it.max())\n    return mz.astype(np.float32), v.astype(np.float32)\n\n\nclass SinEmb(nn.Module):\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\n        super().__init__()\n        n = dim // 2\n        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\n        self.register_buffer('inv', (2 * math.pi) / wav)\n\n    def forward(self, x):\n        a = x.unsqueeze(-1) * self.inv\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h, drop):\n        super().__init__(); self.h = h\n        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)\n        self.n2 = nn.LayerNorm(d)\n        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\n        self.drop = nn.Dropout(drop)\n\n    def forward(self, x, pad):\n        B, N, D = x.shape; y = self.n1(x)\n        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\n        m = (~pad)[:, None, None, :]\n        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)\n        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\n        return x + self.drop(self.ff(self.n2(x)))\n\n\nclass FPNet(nn.Module):\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\n        super().__init__()\n        self.d = d\n        self.mz_emb = SinEmb(d)\n        self.nl_emb = SinEmb(d)\n        self.pk = nn.Linear(2 * d + 1, d)\n        self.prec_emb = SinEmb(d)\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\n        self.gl = nn.Linear(d + 3, d)\n        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\n        self.norm = nn.LayerNorm(d)\n        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\n        B, N = mz.shape\n        nl = (prec[:, None] - mz).clamp(min=0)\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\n        g = self.gl(torch.cat([self.prec_emb(prec),\n                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\n                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\n        x = torch.cat([g.unsqueeze(1), p], 1)\n        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\n        for b in self.blocks: x = b(x, pad)\n        x = self.norm(x)\n        cls = x[:, 0]\n        msk = (~pad[:, 1:]).float().unsqueeze(-1)\n        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\n        return self.head(torch.cat([cls, mean], -1))\n\n\ndef load_fp_models(paths, dev):\n    single, merged = [], []\n    nbits = None\n    for pth in paths:\n        ck = torch.load(pth, map_location='cpu', weights_only=False)\n        net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(dev).eval()\n        net.load_state_dict(ck['model'])\n        (merged if 'merged' in str(pth).replace('\\\\', '/').split('/')[-1] else single).append(net)\n        nbits = ck['nbits']\n    return single, merged, nbits\n\n\ndef merge_peaks(specs):\n    \"\"\"specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum.\"\"\"\n    specs = [(a, b) for a, b in specs if len(a)]\n    if not specs: return np.zeros(0), np.zeros(0)\n    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])\n    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])\n    o = np.argsort(mz); mz, it = mz[o], it[o]\n    keep = np.ones(len(mz), bool)\n    for j in range(1, len(mz)):\n        if mz[j] - mz[j - 1] < 0.005:\n            if it[j] >= it[j - 1]: keep[j - 1] = False\n            else: keep[j] = False\n    return mz[keep], it[keep]\n\n\n@torch.no_grad()\ndef logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):\n    \"\"\"P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode.\"\"\"\n    keep = [i for i, (a, _) in enumerate(P) if len(a)]\n    if not keep: return None\n    P = [P[i] for i in keep]\n    B = len(P); N = max(len(a) for a, _ in P)\n    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)\n    for i, (a, b) in enumerate(P):\n        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False\n    T = lambda x: torch.as_tensor(x, device=dev)\n    args = (T(mz), T(it), T(pad),\n            T(np.asarray(prec, np.float32)[keep]),\n            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX['<unk>']) for a in np.asarray(adduct, object)[keep]])),\n            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),\n            T(np.asarray(ce, np.float32)[keep]),\n            T(np.asarray(mode, np.float32)[keep]))\n    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)\n\n\ndef molecule_logits(models, specs, prec, adduct, instr, ce, mode):\n    \"\"\"prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).\n    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum.\"\"\"\n    single, merged, dev = models\n    out = []\n    if single:\n        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]\n        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)\n        if za is not None: out.append(za)\n    if merged:\n        mz, it = merge_peaks(specs)\n        pm = float(np.median(prec))\n        P = [prep_peaks(mz, it, pm)]\n        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])\n        if zb is not None: out.append(zb)\n    return np.mean(out, axis=0) if out else None", "casmi_engine.py": "\"\"\"End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,\nbut reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).\n\nStages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission\n\"\"\"\nimport os, sys, glob, time, pickle, math\nos.environ.setdefault(\"OPENBLAS_NUM_THREADS\", \"1\")\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\nimport pv\nfrom pv import CFG\n\nPOOL = None\n\n\nclass RANK:\n    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged\n    SEEDS = (0, 1)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n\nT0 = time.time()\nLOCAL = os.environ.get(\"CASMI_LOCAL\") == \"1\"\nROOTS = os.environ[\"CASMI_ROOTS\"].split(os.pathsep) if os.environ.get(\"CASMI_ROOTS\") else (\n    [r\"C:\\Users\\HW-LEE\\Desktop\\CASMI\"] if LOCAL else [\"/kaggle/input\"])\n\n\ndef find(name):\n    for root in ROOTS:\n        hits = glob.glob(os.path.join(root, \"**\", name), recursive=True)\n        if hits: return sorted(hits, key=len)[0]\n    raise FileNotFoundError(name)\n\n\ndef log(msg):\n    print(f\"[{time.time()-T0:6.0f}s] {msg}\", flush=True)\n\n\n# ---------------------------------------------------------------------------------------------\n# library\n# ---------------------------------------------------------------------------------------------\ndef load_library(path):\n    \"\"\"Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB.\"\"\"\n    f = pq.ParquetFile(path)\n    n_rows = f.metadata.num_rows\n    npk = 0\n    for i in range(f.num_row_groups):\n        c = f.read_row_group(i, columns=[\"ms2_mzs\"]).column(0).combine_chunks()\n        npk += len(c.values)\n    off = np.zeros(n_rows + 1, np.int64)\n    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)\n    prec, add, pol, ik, smi = [], [], [], [], []\n    r0 = p0 = 0\n    for i in range(f.num_row_groups):\n        t = f.read_row_group(i, columns=[\"inchikey14\", \"normalized_smiles\", \"adduct\", \"precursor_mz\", \"ionization_mode\",\n                                         \"ms2_mzs\", \"ms2_normalized_intensities\"])\n        mzc = t.column(\"ms2_mzs\").combine_chunks(); itc = t.column(\"ms2_normalized_intensities\").combine_chunks()\n        o = mzc.offsets.to_numpy().astype(np.int64)\n        n = len(o) - 1; k = int(o[-1] - o[0])\n        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])\n        prec.append(t.column(\"precursor_mz\").to_numpy(zero_copy_only=False).astype(np.float64))\n        add += t.column(\"adduct\").cast(pa.string()).to_pylist()\n        pol.append(np.array(t.column(\"ionization_mode\").cast(pa.string()).to_pylist(), dtype=object) == \"positive\")\n        ik += t.column(\"inchikey14\").cast(pa.string()).to_pylist()\n        smi += t.column(\"normalized_smiles\").cast(pa.string()).to_pylist()\n        r0 += n; p0 += k\n        del t, mzc, itc\n    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)\n    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)\n    nm = pv.neutral_mass(prec, add)\n    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol,\n                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))\n\n\n# ---------------------------------------------------------------------------------------------\n# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)\n# ---------------------------------------------------------------------------------------------\n_g = {}\n\n\ndef _fp_init():\n    from rdkit import RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog(\"rdApp.*\")\n    _g[\"bits\"] = np.load(find(\"fp_bits.npy\"))\n    _g[\"m2\"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\n    _g[\"m3\"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\n    _g[\"rk\"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\n\n\ndef fp_and_mass(smi):\n    from rdkit import Chem\n    from rdkit.Chem import MACCSkeys\n    from rdkit.Chem.Descriptors import ExactMolWt\n    if not _g: _fp_init()\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    try:\n        fp = np.concatenate([_g[\"m2\"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g[\"m3\"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g[\"rk\"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g[\"bits\"]]\n        return np.packbits(fp), float(ExactMolWt(m))\n    except Exception:\n        return None\n\n\ndef canon_key(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog(\"rdApp.*\")\n    if \"te\" not in _g: _g[\"te\"] = rdMolStandardize.TautomerEnumerator()\n    try:\n        m = Chem.MolFromSmiles(smi)\n        return Chem.MolToInchiKey(_g[\"te\"].Canonicalize(m))[:14] if m is not None else None\n    except Exception:\n        return None\n\n\ndef build_pool(L, workers):\n    from multiprocessing import Pool as MPool\n    d = os.path.dirname(find(\"coco_fp.npy\"))\n    cm = pickle.load(open(os.path.join(d, \"coco_meta.pkl\"), \"rb\"))\n    co_fp = np.load(os.path.join(d, \"coco_fp.npy\")); co_mass = np.load(os.path.join(d, \"coco_mass.npy\"))\n    co_keys = np.asarray(cm[\"keys\"], dtype=object); co_smi = np.asarray(cm[\"smiles\"], dtype=object)\n    tr = pd.DataFrame({\"ik\": L[\"ik\"], \"smi\": L[\"smi\"]}).dropna().drop_duplicates(\"ik\")\n    tr = tr[~tr.ik.isin(set(co_keys))]\n    log(f\"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}\")\n    with MPool(workers) as mp:\n        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)\n    ok = np.array([r is not None for r in res])\n    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])\n    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])\n    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])\n    good = np.isfinite(mass)\n    o = np.argsort(np.where(good, mass, 1e18), kind=\"mergesort\")[: good.sum()]\n    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])\n    P[\"k2i\"] = pd.Series(np.arange(len(o)), index=P[\"keys\"]); P[\"k2i\"] = P[\"k2i\"][~P[\"k2i\"].index.duplicated()]\n    log(f\"pool {len(o):,} structures\")\n    return P\n\n\ndef pool_fps(idx):\n    return np.unpackbits(np.asarray(POOL[\"fp\"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]\n\n\ndef pool_window(t, ppm):\n    a = np.searchsorted(POOL[\"mass\"], t * (1 - ppm / 1e6), \"left\")\n    b = np.searchsorted(POOL[\"mass\"], t * (1 + ppm / 1e6), \"right\")\n    return np.arange(a, b)\n\n\n# ---------------------------------------------------------------------------------------------\n# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)\n# ---------------------------------------------------------------------------------------------\ndef _reps(L, rows):\n    npk = np.diff(L[\"off\"])[rows]\n    df = pd.DataFrame({\"row\": rows, \"p\": L[\"row_p\"][rows], \"npk\": npk})\n    df = df.sort_values([\"p\", \"npk\", \"row\"], ascending=[True, False, True]).drop_duplicates(\"p\")\n    rep = df.row.values; rep_nm = L[\"nm\"][rep]\n    o = np.argsort(rep_nm, kind=\"mergesort\"); rep = rep[o]\n    roff, rmz, rit = pv.clean_store(rep, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n    return dict(rep=rep, rep_p=L[\"row_p\"][rep], rep_nm=L[\"nm\"][rep], roff=roff, rmz=rmz, rit=rit)\n\n\ndef build_index(L):\n    ok = np.isfinite(L[\"nm\"]) & (L[\"row_p\"] >= 0)\n    rows = np.where(ok)[0]\n    o = np.argsort(L[\"nm\"][rows], kind=\"mergesort\")\n    I = dict(lib_rows=rows[o], lib_nm=L[\"nm\"][rows[o]])\n    I[\"all\"] = _reps(L, rows)\n    I[\"pos\"] = _reps(L, rows[L[\"pol\"][rows] == 1])\n    I[\"neg\"] = _reps(L, rows[L[\"pol\"][rows] == -1])\n    log(f\"index: {len(rows):,} library rows; representatives {len(I['all']['rep']):,}\")\n    return I\n\n\n# ---------------------------------------------------------------------------------------------\n# per-molecule channels (identical to e02_channels.molecule_channels)\n# ---------------------------------------------------------------------------------------------\ndef clean(mz, it):\n    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\n                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n\n\ndef analog_search(R, Q, target):\n    lo = np.searchsorted(R[\"rep_nm\"], target - CFG.ANALOG_WIN, \"left\")\n    hi = np.searchsorted(R[\"rep_nm\"], target + CFG.ANALOG_WIN, \"right\")\n    asim = np.zeros(hi - lo, np.float32)\n    if hi > lo and Q:\n        shift = (target - R[\"rep_nm\"][lo:hi]).astype(np.float32)\n        for qm, qp in Q:\n            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R[\"roff\"], R[\"rmz\"], R[\"rit\"], CFG.MZ_TOL, shift))\n    return R[\"rep_p\"][lo:hi], asim\n\n\ndef top_analogs(p, s):\n    if len(p) == 0: return []\n    df = pd.DataFrame({\"p\": p, \"s\": s}).groupby(\"p\", sort=False).s.max().sort_values(ascending=False, kind=\"mergesort\")\n    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))\n\n\ndef molecule_channels(L, I, specs, precs, pols, target):\n    Q = [clean(a, b) for a, b in specs]\n    keep = [i for i, (a, _) in enumerate(Q) if len(a)]\n    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]\n    tol = target * CFG.PPM_WIN / 1e6\n    lo = np.searchsorted(I[\"lib_nm\"], target - tol, \"left\"); hi = np.searchsorted(I[\"lib_nm\"], target + tol, \"right\")\n    crow = I[\"lib_rows\"][lo:hi]\n    u, inv = np.unique(L[\"row_p\"][crow], return_inverse=True)\n    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)\n    for j, (qm, qp) in enumerate(Q):\n        if len(crow) == 0: break\n        s = pv.search(qm, qp, crow, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                      CFG.INT_POWER, CFG.ENT_WEIGHT)\n        np.maximum.at(M[j], inv, s)\n        sh = (precs[j] - L[\"prec\"][crow]).astype(np.float32)\n        s2 = pv.search_shift_rows(qm, qp, crow, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)\n        np.maximum.at(MS[j], inv, s2)\n    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)\n    z = np.zeros(0, np.float32)\n    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)\n    p, s = analog_search(I[\"all\"], Q, target)\n    ana = top_analogs(p, s)\n    ps, ss = [], []\n    for sign, name in ((1, \"pos\"), (-1, \"neg\")):\n        QQ = [q for q, pl in zip(Q, pols) if pl == sign]\n        if QQ:\n            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)\n    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []\n    return lib, ana, anapol\n\n\ndef parse_ce(v):\n    try:\n        a = np.abs(np.atleast_1d(np.asarray(v, float)))\n        return float(a.mean()) if len(a) else 25.0\n    except Exception:\n        return 25.0\n\n\n\n\n# ---------------------------------------------------------------------------------------------\n# ranking features: prvsiyan's 31 + our blocks (identical to e05_rank.py BLOCKS)\n# ---------------------------------------------------------------------------------------------\ndef _rel(v):\n    v = np.asarray(v, np.float32)\n    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]\n\n\ndef _analog_feats(cfp, ana):\n    nc = len(cfp)\n    if not ana: return np.zeros((nc, 5), np.float32)\n    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)\n    inter = cf @ afp.T\n    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)\n    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)\n    ap = (tan * (w ** 3)[None, :]).max(1)\n    ap6 = (tan * (w ** 6)[None, :]).max(1)\n    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)\n\n\ndef blk_base(r, cfp):\n    ana = r[\"ana\"]\n    afp = pool_fps([p for p, _ in ana]) if ana else None\n    sims = np.array([s for _, s in ana], np.float32)\n    return pv.rank_features(cfp, r[\"lib\"], afp, sims, r[\"zlog\"], r[\"frag\"])\n\n\ndef blk_mass(r, cfp):\n    ppm = (POOL[\"mass\"][r[\"cand\"]] - r[\"target\"]) / r[\"target\"] * 1e6\n    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)\n\n\ndef blk_libx(r, cfp):\n    cnt = np.log1p(r[\"lib_cnt\"])\n    return np.column_stack(_rel(r[\"lib_mean\"]) + _rel(r[\"libsh\"]) + [cnt, cnt - cnt.max()]).astype(np.float32)\n\n\ndef blk_anapol(r, cfp):\n    return _analog_feats(cfp, r[\"anapol\"])\n\n\ndef blk_fragx(r, cfp):\n    return np.column_stack(_rel(r[\"frag_ad\"]) + _rel(r[\"frag_mean\"]) + [r[\"frag_pol\"]]).astype(np.float32)\n\n\nBLOCKS = {\"base\": blk_base, \"mass\": blk_mass, \"libx\": blk_libx, \"anapol\": blk_anapol, \"fragx\": blk_fragx}\n\n\n# ---------------------------------------------------------------------------------------------\n# run\n# ---------------------------------------------------------------------------------------------\ndef compute_channels(L, I, models, te):\n    mols = list(te.groupby(\"molecule_id\", sort=True))\n    recs = []\n    for gi, (mid, sub) in enumerate(mols):\n        specs = []\n        for r in sub.itertuples():\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\n            k_ = a_ <= float(r.precursor_mz) + 2.0\n            a_, b_ = a_[k_], b_[k_]\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\n        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)\n        nms = nms[np.isfinite(nms)]\n        rec = dict(mid=mid, cand=np.zeros(0, np.int64))\n        if len(nms):\n            target = float(np.median(nms))\n            modes = np.where(sub.ionization_mode.astype(str).values == \"positive\", 1.0, -1.0)\n            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)\n            cand = pool_window(target, CFG.PPM_WIN)\n            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)\n            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)\n            import pv_fp\n            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),\n                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),\n                                         ces, modes)\n            u, lmax, lmean, lsh, lcnt = lib\n            pos = {int(x): i for i, x in enumerate(u)}\n            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0\n            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,\n                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),\n                       ana=ana, anapol=anapol)\n            for nm_, arr in ((\"lib\", lmax), (\"lib_mean\", lmean), (\"libsh\", lsh), (\"lib_cnt\", lcnt)):\n                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v\n        recs.append(rec)\n        if gi % 50 == 0: log(f\"  channels {gi}/{len(mols)}\")\n    return mols, recs\n\n\ndef compute_frag(recs, workers):\n    from multiprocessing import Pool as MPool\n    uc = np.unique(np.concatenate([r[\"cand\"] for r in recs]))\n    log(f\"MetFrag-lite on {len(uc):,} candidates\")\n    with MPool(workers) as mp:\n        fr = mp.map(pv.frag_masses_safe, list(POOL[\"smiles\"][uc]), chunksize=16)\n    fmap = dict(zip(uc.tolist(), fr))\n    for r in recs:\n        if not len(r[\"cand\"]): continue\n        peaks = []\n        for a_, b_ in r.pop(\"specs\"):\n            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n        F1 = np.zeros((len(r[\"cand\"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\n        for j, c in enumerate(r[\"cand\"]):\n            fm = fmap[int(c)]\n            for k, (x, y) in enumerate(peaks):\n                F1[j, k] = pv.explain_score(fm, x, y, mode=r[\"mode\"], tol=CFG.MZ_TOL)\n                F2[j, k] = pv.explain_score_adduct(fm, x, y, r[\"adducts\"][k], tol=CFG.MZ_TOL)\n                F3[j, k] = pv.explain_score(fm, x, y, mode=r[\"modes\"][k], tol=CFG.MZ_TOL)\n        r[\"frag\"] = F1.max(1); r[\"frag_ad\"] = F2.max(1); r[\"frag_mean\"] = F3.mean(1); r[\"frag_pol\"] = F3.max(1)\n\n\ndef _hgb(X, Y, W, seed, gbm):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)\n    return m\n\n\ndef fit_rankers(rank_train_path):\n    \"\"\"Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim).\"\"\"\n    z = np.load(rank_train_path, allow_pickle=True)\n    X, Y, S = z[\"X\"], z[\"Y\"], z[\"S\"]\n    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]\n    log(f\"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features\")\n    return rankers, [str(b) for b in z[\"blocks\"]]\n\n\ndef fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):\n    \"\"\"prvsiyan's ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim).\"\"\"\n    z = np.load(pv_rank_path)\n    rankers = [_hgb(z[\"X\"], z[\"Y\"], np.where(z[\"M\"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]\n    log(f\"prvsiyan ranker: {len(rankers)} GBMs on {z['X'].shape[0]:,} rows x {z['X'].shape[1]} features\")\n    return rankers, z[\"X\"].shape[1]\n\n\ndef score_molecules(recs, rankers, blocks, use_fp):\n    out = {}\n    for r in recs:\n        if not len(r[\"cand\"]): continue\n        rr = r if use_fp else dict(r, zlog=None)\n        cfp = pool_fps(r[\"cand\"])\n        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\n        out[r[\"mid\"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)\n    return out\n\n\ndef blend_scores(a, b, wa):\n    \"\"\"Within-molecule rank blend (both inputs are per-candidate probabilities).\"\"\"\n    out = {}\n    for mid in a:\n        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])\n        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]\n    return out\n\n\ndef make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):\n    \"\"\"score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all.\"\"\"\n    from multiprocessing import Pool as MPool\n    cand = {r[\"mid\"]: r[\"cand\"] for r in recs if len(r[\"cand\"])}\n    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind=\"mergesort\")][:topn + 15] for mid, sc in S.items()}\n               for n, S in score_sets.items()}\n    allc = [v for o in ordered.values() for v in o.values()]\n    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)\n    with MPool(workers) as mp:\n        ck = mp.map(canon_key, list(POOL[\"smiles\"][short]), chunksize=64)\n    ckey = dict(zip(short.tolist(), ck))\n    samp = pd.read_csv(sample_path)\n    subs = {}\n    for n, o in ordered.items():\n        rows = []\n        for mid, _ in mols:\n            out, seen = [], set()\n            for c in o.get(mid, []):\n                k = ckey.get(int(c)) or POOL[\"keys\"][c]\n                if k in seen: continue\n                seen.add(k); out.append(POOL[\"smiles\"][c])\n                if len(out) == topn: break\n            while len(out) < topn:\n                out.append(\"CCO\")\n            rows.append((mid, \";\".join(out[:topn])))\n        sub = samp[[\"molecule_id\"]].merge(pd.DataFrame(rows, columns=[\"molecule_id\", \"smiles\"]), on=\"molecule_id\", how=\"left\")\n        sub[\"smiles\"] = sub[\"smiles\"].fillna(\"CCO\")\n        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0\n        assert (sub.smiles.str.split(\";\").map(len) == topn).all()\n        subs[n] = sub\n    return subs\n\n\ndef main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):\n    global POOL\n    import torch\n    import pv_fp\n    L = load_library(train_path); log(f\"library {len(L['prec']):,} spectra\")\n    POOL = build_pool(L, workers)\n    L[\"row_p\"] = POOL[\"k2i\"].reindex(L[\"ik\"]).fillna(-1).values.astype(np.int64)\n    del L[\"ik\"], L[\"smi\"]\n    I = build_index(L)\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)\n    log(f\"fp models: {len(single)} single + {len(merged)} merged on {dev}\")\n    te = pq.read_table(test_path).to_pandas()\n    if limit:\n        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]\n    log(f\"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules\")\n    mols, recs = compute_channels(L, I, (single, merged, dev), te)\n    del L, I\n    compute_frag(recs, workers)\n    ours, blocks = fit_rankers(our_rank_path)\n    s_ours = score_molecules(recs, ours, blocks, use_fp=False)\n    del ours\n    pvr, nfeat = fit_pv_rankers(pv_rank_path)\n    s_pv = score_molecules(recs, pvr, [\"base\"], use_fp=True)\n    del pvr\n    subs = make_submissions(mols, recs, {\"blend\": blend_scores(s_pv, s_ours, w_pv), \"pv\": s_pv, \"ours\": s_ours},\n                            sample_path, workers)\n    log(\"submissions ready\")\n    return subs, recs", "metric.py": "\"\"\"Kaggle scoring for CASMI: Mean Reciprocal Rank @ max_rank (MRR@25 by default).\n\nSubmission is one row per molecule, `molecule_id,smiles`, where `smiles` holds up to `max_rank`\ncandidate structures joined by semicolons, best guess first:\n\n    molecule_id,smiles\n    m_0014ef,CCO;CC(=O)O;c1ccccc1\n\nEach molecule scores 1/position of the first guess whose tautomer-canonical InChIKey14 matches\nthe answer, else 0. There is exactly one correct structure per molecule, so nothing accumulates\npast the first hit -- this is reciprocal rank, not average precision, and it stays reciprocal\nrank even if `canonical_ik14s_all` is later extended to accept several indistinguishable\nisomers, because the loop breaks on the first match. `max_rank` is a metric parameter set on\nthe competition page.\nInlines RDKit canonicalization to run standalone on Kaggle.\n\"\"\"\n\nimport importlib\nimport os\nimport subprocess\nimport sys\nimport warnings\n\nimport numpy as np\nimport pandas as pd\n\n# Solution key canonicalized with RDKit 2026.03.3. RDKit's TautomerEnumerator output has changed\n# between releases, so this pin decides which submissions match; it is not a convenience.\n# https://pypi.org/project/rdkit/2026.3.3/\nEXPECTED_RDKIT_VERSION = '2026.03.3'\n# PyPI spells the same release 2026.3.3 -- PEP 440 strips the leading zero -- while\n# rdkit.__version__ reports it zero-padded. Both strings appear below on purpose.\nRDKIT_REQUIREMENT = 'rdkit==2026.3.3'\n# RDKit is not in docker-python, so the wheel ships as a competition input and is installed\n# offline at scoring time. Wheel dataset: https://www.kaggle.com/datasets/<owner>/rdkit-2026-3-3-wheel\nRDKIT_WHEEL_DIR = '/kaggle/input/datasets/metric/rdkit-2026-3-3-wheel'\n\n# Bound by install_dependencies(); importing this module must not require RDKit, because the\n# orchestrator imports it before anything has had a chance to install.\nChem = None\n_tautomer_canonicalizer = None\n\n\nclass HostVisibleError(Exception):\n    pass\n\n\ndef install_dependencies() -> None:\n    \"\"\"Make RDKit importable, installing it from the bundled wheel if the image lacks it.\n\n    Idempotent and cheap on repeat calls. Called at the top of score() rather than at import time:\n    validate_local_metrics and the orchestrator both import this module first, and an import-time\n    install would run before the competition inputs are mounted.\n    \"\"\"\n    global Chem, _tautomer_canonicalizer\n    if Chem is not None:\n        return\n\n    try:\n        import rdkit\n    except ImportError:\n        if not os.path.isdir(RDKIT_WHEEL_DIR):\n            raise HostVisibleError(\n                f'{RDKIT_WHEEL_DIR} does not exist, so {RDKIT_REQUIREMENT} cannot be installed. '\n                f'Attach the wheel dataset to the metric notebook as an input.'\n            )\n        # pip rather than uv: pip always installs into the interpreter running it, whereas\n        # `uv pip install` wants a virtualenv and exits 2 when there isn't one. --no-deps because\n        # rdkit only needs numpy and Pillow, both already in the image; without it the resolver\n        # goes looking for them and --no-index leaves it nowhere to look.\n        command = [\n            sys.executable,\n            '-m',\n            'pip',\n            'install',\n            '--no-deps',\n            '--no-index',\n            f'--find-links={RDKIT_WHEEL_DIR}',\n            RDKIT_REQUIREMENT,\n        ]\n        # Captured, not inherited: a bare check=True reports only the exit status, which says\n        # nothing about whether the wheel was missing, unresolvable, or wrong for this Python.\n        result = subprocess.run(command, capture_output=True, text=True, check=False)\n        if result.returncode != 0:\n            available = sorted(os.listdir(RDKIT_WHEEL_DIR))\n            raise HostVisibleError(\n                f'{\" \".join(command)} exited {result.returncode}. '\n                f'Python {sys.version_info.major}.{sys.version_info.minor}; '\n                f'{RDKIT_WHEEL_DIR} contains {available}. '\n                f'stderr: {result.stderr[-1500:]} stdout: {result.stdout[-1500:]}'\n            )\n        importlib.invalidate_caches()\n        try:\n            import rdkit\n        except ImportError as err:\n            raise HostVisibleError(f'{RDKIT_REQUIREMENT} installed but still not importable: {err}')\n\n    from rdkit import Chem as _Chem\n    from rdkit import RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n\n    RDLogger.DisableLog('rdApp.*')\n    if rdkit.__version__ != EXPECTED_RDKIT_VERSION:\n        warnings.warn(\n            f'RDKit version mismatch: solution={EXPECTED_RDKIT_VERSION}, scoring={rdkit.__version__}. Canonicalization may differ.',\n            UserWarning,\n        )\n    Chem = _Chem\n    _tautomer_canonicalizer = rdMolStandardize.TautomerEnumerator()\n\n\ndef canonicalize_smiles_to_ik14(smiles: str) -> str:\n    \"\"\"Canonicalize SMILES to InChIKey14, or None if parsing fails.\"\"\"\n    if not isinstance(smiles, str):\n        return None\n    try:\n        mol = Chem.MolFromSmiles(smiles)\n        if mol is None:\n            return None\n        canonical_mol = _tautomer_canonicalizer.Canonicalize(mol)\n        inchikey = Chem.MolToInchiKey(canonical_mol)\n        return inchikey[:14]\n    # RDKit's MolSanitizeException family subclasses ValueError; RuntimeError covers the C++\n    # layer and TypeError covers Boost.Python argument errors. One participant's malformed\n    # SMILES must never take the whole scoring run down, so a failure here is just a non-match.\n    except (ValueError, RuntimeError, TypeError):\n        return None\n\n\ndef heavy_atom_composition(smiles: str):\n    \"\"\"Heavy-atom element counts (excluding H, ignoring charge) as a hashable key, or None\n    if parsing fails.\n\n    Cheap (no tautomer enumeration) -- used to gate the expensive canonicalization. Anything\n    that shares an InChIKey14 with an accepted structure has the same heavy-atom skeleton, so\n    a heavy-atom-composition mismatch lets us skip canonicalization with no change to any\n    score. We deliberately do NOT gate on full molecular formula: protomers / charge variants\n    (e.g. a zwitterion C(=O)[O-]...[N+] vs its neutral-acid protomer C(=O)O...[N+]) share an\n    InChIKey14 but differ in H count and charge, so a formula gate would wrongly reject them.\n    \"\"\"\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\n    if mol is None:\n        return None\n    counts = {}\n    for atom in mol.GetAtoms():\n        sym = atom.GetSymbol()\n        if sym != 'H':\n            counts[sym] = counts.get(sym, 0) + 1\n    return tuple(sorted(counts.items()))\n\n\nclass ParticipantVisibleError(Exception):\n    pass\n\n\ndef score(\n    solution: pd.DataFrame,\n    submission: pd.DataFrame,\n    row_id_column_name: str,\n    max_rank: int = 25,\n) -> float:\n    \"\"\"Score CASMI submissions using mean reciprocal rank.\n\n    max_rank: how many ranked guesses each molecule may carry. Set on the competition page.\n\n    Examples\n    --------\n    >>> import pandas as pd\n    >>> solution = pd.DataFrame({\n    ...     'molecule_id': ['m_1', 'm_2'],\n    ...     'smiles': ['CCO', 'c1ccccc1'],\n    ...     'Usage': ['Public', 'Public'],\n    ... })\n\n    Position in the semicolon-separated list is the rank: ethanol is guessed first and scores\n    1.0, benzene is guessed second and scores 1/2, so the mean is 0.75.\n\n    >>> submission = pd.DataFrame({\n    ...     'molecule_id': ['m_1', 'm_2'],\n    ...     'smiles': ['CCO;CC(=O)O', 'CC(=O)O;c1ccccc1'],\n    ... })\n    >>> score(solution, submission, 'molecule_id')\n    0.75\n\n    A molecule with no correct guess scores zero, and only the first correct guess counts, so\n    repeating it gains nothing.\n\n    >>> score(solution, pd.DataFrame({\n    ...     'molecule_id': ['m_1', 'm_2'],\n    ...     'smiles': ['CC(=O)O', 'c1ccccc1;c1ccccc1;c1ccccc1'],\n    ... }), 'molecule_id')\n    0.5\n\n    Matching is on the tautomer-canonical InChIKey14, so stereochemistry is ignored: the\n    submission below omits the stereocentre the answer specifies and still scores.\n\n    >>> score(\n    ...     pd.DataFrame({'molecule_id': ['m_1'], 'smiles': ['C[C@H](N)C(=O)O']}),\n    ...     pd.DataFrame({'molecule_id': ['m_1'], 'smiles': ['CC(N)C(=O)O']}),\n    ...     'molecule_id',\n    ... )\n    1.0\n\n    Malformed submissions raise ParticipantVisibleError.\n\n    >>> try:\n    ...     score(solution, pd.DataFrame({\n    ...         'molecule_id': ['m_1', 'm_1'], 'smiles': ['CCO', 'CCO'],\n    ...     }), 'molecule_id')\n    ... except ParticipantVisibleError as err:\n    ...     print(err)\n    Each molecule_id must appear exactly once\n\n    >>> try:\n    ...     score(solution, pd.DataFrame({\n    ...         'molecule_id': ['m_1'], 'smiles': [';'.join(['CCO'] * 3)],\n    ...     }), 'molecule_id', max_rank=2)\n    ... except ParticipantVisibleError as err:\n    ...     print(err)\n    At most 2 semicolon-separated guesses per molecule_id, got 3 for m_1\n\n    An answer the canonicalizer cannot parse is a broken solution file, not a zero score.\n\n    >>> try:\n    ...     score(\n    ...         pd.DataFrame({'molecule_id': ['m_1'], 'smiles': ['not a molecule']}),\n    ...         pd.DataFrame({'molecule_id': ['m_1'], 'smiles': ['CCO']}),\n    ...         'molecule_id',\n    ...     )\n    ... except HostVisibleError as err:\n    ...     print(err)\n    Solution SMILES for molecule_id m_1 could not be parsed: 'not a molecule'\n    \"\"\"\n    install_dependencies()\n    if not isinstance(max_rank, (int, np.integer)) or max_rank < 1:\n        raise ValueError(f'max_rank must be a positive integer, got {max_rank!r}')\n    required = ['molecule_id', 'smiles']\n    if not all(c in submission.columns for c in required):\n        raise ParticipantVisibleError(f'Missing columns: {required}')\n    if len(submission) == 0:\n        raise ParticipantVisibleError('Submission is empty')\n    if submission[required].isna().any().any():\n        raise ParticipantVisibleError('NaN in required columns')\n    if 'smiles' not in solution.columns:\n        raise ParticipantVisibleError('Solution missing smiles column')\n    if submission['molecule_id'].duplicated().any():\n        raise ParticipantVisibleError('Each molecule_id must appear exactly once')\n\n    # One row per molecule; rank is position in the list, so there is no rank column to validate\n    # and no way to claim two guesses share a rank. A semicolon cannot occur inside a valid SMILES\n    # -- RDKit rejects \"CCO;CC(=O)O\" -- so splitting on it can never cut a legitimate guess in half.\n    # Whitespace would not be safe here: RDKit reads \"CCO ethanol\" as ethanol with a name attached.\n    guesses_by_molecule = {}\n    for molecule_id, raw_guesses in zip(submission['molecule_id'], submission['smiles']):\n        parsed = [candidate.strip() for candidate in str(raw_guesses).split(';')]\n        parsed = [candidate for candidate in parsed if candidate]\n        if not parsed:\n            raise ParticipantVisibleError(f'No guesses for molecule_id {molecule_id}')\n        if len(parsed) > max_rank:\n            raise ParticipantVisibleError(f'At most {max_rank} semicolon-separated guesses per molecule_id, got {len(parsed)} for {molecule_id}')\n        guesses_by_molecule[molecule_id] = parsed\n\n    # Cheap heavy-atom-composition gate (computed up front; no tautomer enumeration).\n    gate_keys = {s: heavy_atom_composition(s) for guesses in guesses_by_molecule.values() for s in guesses}\n\n    scores = []\n    unknown_ids = set(submission['molecule_id']) - set(solution['molecule_id'])\n    if unknown_ids:\n        print(f'Unknown molecule_ids: {sorted(unknown_ids)[:5]}{\"...\" if len(unknown_ids) > 5 else \"\"}')\n\n    # Per molecule: the accepted InChIKey14 set and the heavy-atom compositions those accepted\n    # structures can have (one composition in practice; built as a set to be safe). The keys are\n    # derived from the answer SMILES through the same canonicalization a submission goes through,\n    # rather than stored alongside them -- one source of truth, and the answer key cannot drift out\n    # of step with the canonicalizer the way a precomputed column silently would.\n    sol_dict = {}\n    for _, row in solution.iterrows():\n        molecule_id = row['molecule_id']\n        accept_smiles = row['smiles'] if isinstance(row['smiles'], str) else ''\n        candidates = [s for s in accept_smiles.split(',') if s]\n        if not candidates:\n            raise HostVisibleError(f'Solution has no answer SMILES for molecule_id {molecule_id}')\n        accept_ik14s = {k for k in map(canonicalize_smiles_to_ik14, candidates) if k}\n        accept_keys = {k for k in map(heavy_atom_composition, candidates) if k}\n        # Skipping an unparseable answer would quietly drop the molecule from the mean, so a\n        # corrupt key would read as a lower score rather than as a broken solution file.\n        if not accept_ik14s or not accept_keys:\n            raise HostVisibleError(f'Solution SMILES for molecule_id {molecule_id} could not be parsed: {accept_smiles!r}')\n        sol_dict[molecule_id] = (accept_ik14s, accept_keys)\n\n    # Memoize canonicalization so a repeated SMILES is only canonicalized once.\n    canon_cache = {}\n\n    def canon(smiles):\n        if smiles not in canon_cache:\n            canon_cache[smiles] = canonicalize_smiles_to_ik14(smiles)\n        return canon_cache[smiles]\n\n    for mol_id, (acceptable, accept_keys) in sol_dict.items():\n        ap = 0.0\n        for rank, guess in enumerate(guesses_by_molecule.get(mol_id, []), 1):\n            # Gate: skip the costly tautomer canonicalization when heavy-atom composition\n            # rules out a match. Not guarded on `accept_keys` being populated -- the solution\n            # parsing above raises if it is empty, so a silently disabled gate cannot happen.\n            if gate_keys[guess] not in accept_keys:\n                continue\n            if canon(guess) in acceptable:\n                ap = 1.0 / rank\n                break\n        scores.append(ap)\n\n    return float(np.mean(scores))\n\n", "harness.py": "\"\"\"Local validation harness for the two-ranker engine.\n\nSimulates the hidden test on held-out timsTOF molecules, twice per molecule:\n  c1  Class-1 simulation: the query's own source library is removed from the spectral library,\n      other libraries' spectra of the same structure stay (cross-library library search).\n  c2  Class-2 simulation: every spectrum of the structure is removed from the library; the\n      structure stays in the candidate pool (as a training structure: the shipped COCONUT half\n      has all training structures removed, so its own coverage cannot be checked here -- prvsiyan\n      measured 99.6% for enveda-np-examples). c2 MRR is therefore conditional on pool membership.\n\n  python harness.py build [--set np] [--limit N]   # heavy: library, pool, channels, MetFrag -> cache\n  python harness.py score [--set np]               # light: rankers, blend, metric, report\n\nThe engine modules (pv, pv_fp, casmi_engine) are the notebook's code; this file only wraps them.\n\"\"\"\nimport os, sys, time, pickle, argparse\nHERE = os.path.dirname(os.path.abspath(__file__))\nROOT = os.path.dirname(HERE)\nDATA = os.path.join(ROOT, \"data\")\nCACHE = os.path.join(DATA, \"cache\")\nos.environ.setdefault(\"CASMI_ROOTS\", DATA)\nsys.path.insert(0, HERE)\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\n\nTRAIN = os.path.join(DATA, \"comp\", \"train.parquet\")\nTEST_ADDUCTS = {\"[M+H]+\", \"[M+NH4]+\", \"[M-H2O+H]+\", \"[M-2H2O+H]+\", \"[M+Na]+\", \"[M+K]+\",\n                \"[M-H]-\", \"[M-H2O-H]-\", \"[M+CH2O2-H]-\", \"[M+Cl]-\"}\nTEST_COLS = [\"molecule_id\", \"spectrum_id\", \"ms2_mzs\", \"ms2_normalized_intensities\", \"base_peak_intensity\", \"adduct\",\n             \"ionization_mode\", \"instrument_type\", \"precursor_mz\", \"collision_energy_ev\"]\n# query sets: source library, and how many structures to sample (None = all)\nQUERY_SETS = {\"np\": dict(lib=\"enveda-np-examples\", n=None)}\n# prvsiyan's leaderboard calibration: LB ~ A*c1 + B*c2 (A = Class-1 share; B = Class-2 share x pool recall,\n# refitted by him to 0.271 as the ranker improved; megayak used 0.220)\nLB_A, LB_B = 0.162, 0.271\nPER_CAND = (\"cand\", \"lib\", \"lib_mean\", \"libsh\", \"lib_cnt\", \"frag\", \"frag_ad\", \"frag_mean\", \"frag_pol\")\n\n\ndef log(msg, t0=[time.time()]):\n    print(f\"[{time.time() - t0[0]:6.0f}s] {msg}\", flush=True)\n\n\ndef cpath(*parts):\n    os.makedirs(CACHE, exist_ok=True)\n    return os.path.join(CACHE, *parts)\n\n\n# ---------------------------------------------------------------------------------------------\n# queries\n# ---------------------------------------------------------------------------------------------\ndef build_queries(set_name, limit=0, seed=0):\n    \"\"\"Test-like queries from one source library: test adducts only, the host's cleaning rules,\n    one molecule per inchikey14 (all of its spectra from that library).\"\"\"\n    q = QUERY_SETS[set_name]\n    cols = [c for c in TEST_COLS if c not in (\"molecule_id\", \"spectrum_id\")] + [\"inchikey14\", \"normalized_smiles\",\n                                                                                  \"precursor_error_ppm\"]\n    t = pq.read_table(TRAIN, columns=cols, filters=[(\"ingest_lib\", \"=\", q[\"lib\"])]).to_pandas()\n    t[\"spectrum_id\"] = [f\"s_{i}\" for i in range(len(t))]\n    n0 = len(t)\n    t = t[t.adduct.astype(str).isin(TEST_ADDUCTS)]\n    t = t[t.base_peak_intensity.isna() | (t.base_peak_intensity >= 1000)]\n    t = t[t.precursor_error_ppm.isna() | (t.precursor_error_ppm.abs() <= 20)]\n    iks = np.array(sorted(t.inchikey14.unique()))\n    if q[\"n\"] and len(iks) > q[\"n\"]:\n        iks = np.sort(np.random.default_rng(seed).choice(iks, q[\"n\"], replace=False))\n    if limit:\n        iks = iks[:limit]\n    t = t[t.inchikey14.isin(set(iks))].copy()\n    t[\"molecule_id\"] = \"q_\" + t.inchikey14.astype(str)\n    truth = t.drop_duplicates(\"molecule_id\")[[\"molecule_id\", \"inchikey14\", \"normalized_smiles\"]].sort_values(\"molecule_id\")\n    log(f\"queries[{set_name}]: {n0:,} library spectra -> {len(t):,} test-like spectra / {len(truth)} molecules \"\n        f\"(spectra/molecule median {t.groupby('molecule_id').size().median():.0f})\")\n    return t.reset_index(drop=True), truth.reset_index(drop=True)\n\n\n# ---------------------------------------------------------------------------------------------\n# heavy stage\n# ---------------------------------------------------------------------------------------------\ndef load_pool(E, L, workers):\n    \"\"\"E.build_pool, cached (fingerprinting 276k training structures is the slow part).\"\"\"\n    f = cpath(\"pool.pkl\")\n    if os.path.exists(f):\n        P = pickle.load(open(f, \"rb\")); P[\"fp\"] = np.load(cpath(\"pool_fp.npy\"), mmap_mode=\"r\")\n        log(f\"pool {len(P['mass']):,} structures (cached)\")\n    else:\n        P = E.build_pool(L, workers)\n        np.save(cpath(\"pool_fp.npy\"), P[\"fp\"])\n        pickle.dump({k: v for k, v in P.items() if k != \"fp\"}, open(f, \"wb\"), protocol=4)\n    return P\n\n\ndef build(args):\n    import casmi_engine as E, pv_fp, torch\n    torch.set_num_threads(max(1, args.workers // 2))\n    te, truth = build_queries(args.set, args.limit)\n    L = E.load_library(TRAIN); log(f\"library {len(L['prec']):,} spectra\")\n    lib = pq.read_table(TRAIN, columns=[\"ingest_lib\"]).column(0).cast(pa.string()).to_numpy(zero_copy_only=False)\n    E.POOL = load_pool(E, L, args.workers)\n    L[\"row_p\"] = E.POOL[\"k2i\"].reindex(L[\"ik\"]).fillna(-1).values.astype(np.int64)\n    ik = L.pop(\"ik\"); del L[\"smi\"]\n    base_row_p = L[\"row_p\"].copy()\n    import glob\n    single, merged, _ = pv_fp.load_fp_models(sorted(glob.glob(os.path.join(DATA, \"fpmodels\", \"fp_*.pt\"))), \"cpu\")\n    models = (single, merged, \"cpu\")\n    log(f\"fp models: {len(models[0])} single + {len(models[1])} merged\")\n\n    qlib = QUERY_SETS[args.set][\"lib\"]\n    held = set(truth.inchikey14)\n    for sim in args.sims:\n        t0 = time.time()\n        L[\"row_p\"] = base_row_p.copy()\n        if sim == \"c1\":\n            mask = lib == qlib\n        else:\n            mask = np.isin(ik, list(held))\n        L[\"row_p\"][mask] = -1\n        log(f\"[{sim}] masked {mask.sum():,} library rows\")\n        I = E.build_index(L)\n        mols, recs = E.compute_channels(L, I, models, te)\n        del I\n        E.compute_frag(recs, args.workers)\n        pickle.dump(dict(mids=[m for m, _ in mols], recs=recs, truth=truth, set=args.set, sim=sim,\n                         seconds=time.time() - t0), open(cpath(f\"recs_{args.set}_{sim}.pkl\"), \"wb\"), protocol=4)\n        log(f\"[{sim}] done in {time.time() - t0:.0f}s\")\n\n\n# ---------------------------------------------------------------------------------------------\n# light stage\n# ---------------------------------------------------------------------------------------------\ndef canon_map(E, idx, workers):\n    \"\"\"Tautomer-canonical InChIKey14 (the scorer's key) for pool indices, cached across runs.\"\"\"\n    from multiprocessing import Pool as MPool\n    f = cpath(\"canon.pkl\")\n    cm = pickle.load(open(f, \"rb\")) if os.path.exists(f) else {}\n    todo = [int(i) for i in np.unique(idx) if int(i) not in cm]\n    if todo:\n        with MPool(workers) as mp:\n            ks = mp.map(E.canon_key, list(E.POOL[\"smiles\"][todo]), chunksize=64)\n        cm.update(zip(todo, ks)); pickle.dump(cm, open(f, \"wb\"), protocol=4)\n    return cm\n\n\ndef reciprocal_ranks(sub, truth_key, smi_key, metric):\n    \"\"\"Per-molecule reciprocal rank, same rule as the official metric (first tautomer-canonical ik14 hit).\n    smi_key caches the canonical key of pool SMILES; anything else is canonicalized by the metric itself.\"\"\"\n    rr = {}\n    for mid, s in zip(sub.molecule_id, sub.smiles):\n        k = truth_key[mid]; rr[mid] = 0.0\n        for r, g in enumerate([x.strip() for x in s.split(\";\") if x.strip()][:25], 1):\n            if g not in smi_key: smi_key[g] = metric.canonicalize_smiles_to_ik14(g)\n            if smi_key[g] == k:\n                rr[mid] = 1.0 / r; break\n    return rr\n\n\ndef boot_se(v, n=2000, seed=0):\n    v = np.asarray(v); rng = np.random.default_rng(seed)\n    return float(np.std([v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)]))\n\n\ndef channel_scores(E, recs):\n    \"\"\"Single-channel rankings from prvsiyan's 31 base features (no ranker).\"\"\"\n    cols = {\"lib\": 0, \"analog\": 5, \"fpnet f.z\": 15, \"metfrag\": 21}\n    out = {k: {} for k in cols}\n    out[\"lib_shift\"] = {}\n    for r in recs:\n        if not len(r[\"cand\"]): continue\n        X = E.blk_base(r, E.pool_fps(r[\"cand\"]))\n        for k, c in cols.items():\n            out[k][r[\"mid\"]] = X[:, c] + 1e-6 * X[:, 5]  # analog as tie-break\n        out[\"lib_shift\"][r[\"mid\"]] = np.asarray(r[\"libsh\"], np.float32) + 1e-6 * X[:, 5]\n    return out\n\n\ndef pv_rows(path, drop_groups):\n    z = np.load(path)\n    keep = ~np.isin(z[\"G\"], list(drop_groups))\n    return z[\"X\"][keep], z[\"Y\"][keep], z[\"M\"][keep]\n\n\ndef my_rows_(path, drop_sets):\n    z = np.load(path, allow_pickle=True)\n    keep = ~np.isin(z[\"Q\"], [list(z[\"sets\"]).index(s) for s in drop_sets])\n    return z[\"X\"][keep], z[\"Y\"][keep], z[\"S\"][keep], [str(b) for b in z[\"blocks\"]]\n\n\ndef fit_A(E, X, Y, M, mult=1.0):\n    \"\"\"prvsiyan's ranker recipe: 2 class priors x 4 seeds (M = 0 marks Class-1 simulation rows).\n    mult: extra per-row weight (used to up-weight our in-domain rows).\"\"\"\n    return [E._hgb(X, Y, np.where(M == 0, w1, 1.0 - w1) * mult, sd, E.CFG.GBM) for w1 in (0.30, 0.60) for sd in (0, 1, 2, 3)]\n\n\ndef fit_B(E, X, Y, S, mult=1.0):\n    \"\"\"megayak's ranker recipe: RANK.W_A priors x RANK.SEEDS (S = 0 marks Class-1 simulation rows).\"\"\"\n    return [E._hgb(X, Y, np.where(S == 0, wA, 1.0 - wA) * mult, sd, E.RANK.GBM) for wA in E.RANK.W_A for sd in E.RANK.SEEDS]\n\n\ndef own_rows(E, runs, blocks, use_fp):\n    \"\"\"Ranker rows from our own cached simulations, built by the same feature code the engine scores with.\n    Returns X, Y, M (0 = Class-1 sim), and the molecule id of every row (for folds).\"\"\"\n    Xs, Ys, Ms, Ids = [], [], [], []\n    for sim, R in runs.items():\n        for r in R[\"recs\"]:\n            if not len(r[\"cand\"]) or r[\"truth_idx\"] < 0: continue\n            rr = r if use_fp else dict(r, zlog=None)\n            cfp = E.pool_fps(r[\"cand\"])\n            X = np.hstack([E.BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\n            y = np.zeros(len(X), np.float32); y[r[\"truth_idx\"]] = 1\n            Xs.append(X); Ys.append(y); Ms.append(np.full(len(X), 0 if sim == \"c1\" else 1)); Ids.append([r[\"mid\"]] * len(X))\n    return np.vstack(Xs), np.concatenate(Ys), np.concatenate(Ms), np.concatenate(Ids)\n\n\ndef np_groups_in_pv_rows(E, recs_c2, path):\n    \"\"\"prvsiyan's rank_train.npz has no query-set label, but its Class-2 groups for enveda-np-examples\n    should reproduce our own Class-2 features. Match each of our molecules to the pv group whose\n    true-candidate row is nearest (on channel columns that do not depend on the FP model).\"\"\"\n    z = np.load(path)\n    X, Y, G, M = z[\"X\"], z[\"Y\"], z[\"G\"], z[\"M\"]\n    pos = (Y == 1) & (M == 1)\n    gx, gX = G[pos], X[pos][:, [5, 6, 9, 10, 11, 12, 13, 14]]\n    matched, dist = [], []\n    for r in recs_c2:\n        if not len(r[\"cand\"]): continue\n        tk = r.get(\"truth_idx\", -1)\n        if tk < 0: continue\n        x = E.blk_base(dict(r, zlog=None), E.pool_fps(r[\"cand\"]))[tk][[5, 6, 9, 10, 11, 12, 13, 14]]\n        d = np.abs(gX - x).max(1); j = int(np.argmin(d))\n        matched.append(int(gx[j])); dist.append(float(d[j]))\n    return np.array(matched), np.array(dist)\n\n\ndef _E():\n    import casmi_engine; return casmi_engine\n\n\ndef _pv():\n    import pv; return pv\n\n\n# ---------------------------------------------------------------------------------------------\n# stacker: in-domain (timsTOF) second stage over both rankers + library evidence.\n# Measured 2026-09-24: +0.011 pred LB with the original candidate order, -0.004 +/- 0.004 once candidates are\n# shuffled -- the whole gain was the position leak (see --shuffle). Kept as a harness method, not shipped.\n# ---------------------------------------------------------------------------------------------\nclass E_STACK:\n    W_C1, W_C2 = 0.162, 0.271  # LB calibration shares: Class-1 / Class-2 simulation rows\n    SEEDS = (0, 1)\n    GBM = dict(max_depth=4, max_iter=300, learning_rate=0.05, min_samples_leaf=80)\n\n\ndef stack_features(r, a, b, w_pv=0.88):\n    \"\"\"Per-candidate rows for the stacker. a, b: Ranker A / Ranker B probabilities for r['cand'].\n    Library evidence enters per candidate and as molecule context: a Class-2 answer never has library spectra,\n    a Class-1 answer usually has the strongest (shifted) match and many spectra.\"\"\"\n    n = len(a); rn = lambda v: 1.0 - _pv()._rank_norm(v)\n    ls, lib, cnt = np.asarray(r[\"libsh\"], np.float32), np.asarray(r[\"lib\"], np.float32), np.log1p(r[\"lib_cnt\"])\n    bl = w_pv * rn(a) + (1.0 - w_pv) * rn(b)\n    j, kb = int(np.argmax(ls)), int(np.argmax(bl))\n    per = [a, b, rn(a), rn(b), a - a.max(), b - b.max(), rn(bl), ls, rn(ls), ls - ls.max(), lib, lib - lib.max(),\n           cnt, cnt - cnt.max(), (np.arange(n) == j).astype(np.float32)]\n    mol = [ls.max(), np.sort(ls)[-2] if n > 1 else 0.0, cnt.max(), cnt[j], a.max(), b.max(), rn(bl)[j], ls[kb],\n           float(j == kb), float(n)]\n    return np.column_stack(per + [np.full(n, v) for v in mol]).astype(np.float32)\n\n\ndef fit_stacker(X, Y, S):\n    \"\"\"S = 0 Class-1 simulation rows, 1 = Class-2 (weighted by their LB shares).\"\"\"\n    W = np.where(S == 0, E_STACK.W_C1, E_STACK.W_C2)\n    return [_E()._hgb(X, Y, W, sd, E_STACK.GBM) for sd in E_STACK.SEEDS]\n\n\ndef score_stacker(recs, stackers, s_a, s_b):\n    out = {}\n    for r in recs:\n        mid = r[\"mid\"]\n        if mid not in s_a: continue\n        X = stack_features(r, s_a[mid], s_b[mid])\n        out[mid] = np.mean([m.predict_proba(X)[:, 1] for m in stackers], axis=0) + 1e-6 * s_a[mid]\n    return out\n\n\ndef score(args):\n    import casmi_engine as E, metric\n    metric.install_dependencies()\n    E.POOL = pickle.load(open(cpath(\"pool.pkl\"), \"rb\")); E.POOL[\"fp\"] = np.load(cpath(\"pool_fp.npy\"), mmap_mode=\"r\")\n    runs = {s: pickle.load(open(cpath(f\"recs_{args.set}_{s}.pkl\"), \"rb\")) for s in args.sims\n            if os.path.exists(cpath(f\"recs_{args.set}_{s}.pkl\"))}\n    truth = next(iter(runs.values()))[\"truth\"]\n    truth_key = {m: metric.canonicalize_smiles_to_ik14(s) for m, s in zip(truth.molecule_id, truth.normalized_smiles)}\n    cm = canon_map(E, np.concatenate([r[\"cand\"] for R in runs.values() for r in R[\"recs\"]]), args.workers)\n    smi_key = {E.POOL[\"smiles\"][i]: k for i, k in cm.items()}\n    if args.shuffle:\n        # The pool is mass-sorted with a stable sort: same-formula isomers keep COCONUT-before-training order, and a\n        # simulated answer is always a training structure. Shuffling candidates removes that position signal.\n        rng = np.random.default_rng(args.shuffle)\n        for R in runs.values():\n            for r in R[\"recs\"]:\n                p = rng.permutation(len(r[\"cand\"]))\n                for k in PER_CAND:\n                    if k in r: r[k] = np.asarray(r[k])[p]\n    for R in runs.values():\n        for r in R[\"recs\"]:\n            hit = [i for i, c in enumerate(r[\"cand\"]) if cm.get(int(c)) == truth_key[r[\"mid\"]]]\n            r[\"truth_idx\"] = hit[0] if hit else -1\n\n    sfx = (f\"_shuf{args.shuffle}\" if args.shuffle else \"\") + (f\"_npw{args.np_weight:g}\" if args.np_weight != 1 else \"\")\n    pv_rank = E.find(\"rank_train.npz\"); my_rows = E.find(\"sim_rank_rows_nofp.npz\")\n    drop_A = set()\n    if \"c2\" in runs:\n        g, d = np_groups_in_pv_rows(E, runs[\"c2\"][\"recs\"], pv_rank)\n        exact = d < 1e-2\n        drop_A = set(g[exact].tolist())\n        # every confident match lands in groups 0..249 of 819: pv wrote the enveda-np-examples block first,\n        # so hold out that whole block rather than only the molecules we could re-identify\n        if args.set == \"np\" and drop_A and max(drop_A) < len(truth):\n            drop_A = set(range(len(truth)))\n        log(f\"Ranker A rows: matched {exact.sum()}/{len(d)} of our Class-2 molecules to pv groups (max-abs dist < 1e-2; \"\n            f\"{len(drop_A)} distinct groups, ids {min(drop_A, default=-1)}..{max(drop_A, default=-1)}); \"\n            f\"distance quantiles 50/90/max {np.quantile(d, [.5, .9, 1]).round(4).tolist()}\")\n    t0 = time.time()\n\n    def cached(key, fn):\n        \"\"\"Fitted GBMs keyed by what they were trained on (row file, dropped rows, engine GBM settings).\"\"\"\n        import hashlib\n        h = hashlib.md5(repr((key, E.CFG.GBM, E.RANK.GBM, E.RANK.W_A, E.RANK.SEEDS)).encode()).hexdigest()[:12]\n        f = cpath(f\"rankers_{h}.pkl\")\n        if os.path.exists(f): return pickle.load(open(f, \"rb\"))\n        v = fn(); pickle.dump(v, open(f, \"wb\"), protocol=4); return v\n\n    # As shipped: both row files contain these very np molecules (in-sample, optimistic reference).\n    XA, YA, MA = pv_rows(pv_rank, set())\n    XB, YB, SB, blocks = my_rows_(my_rows, [])\n    shipped = {\"A as shipped (in-sample)\": (cached((\"A-shipped\", ()), lambda: fit_A(E, XA, YA, MA)), [\"base\"], True),\n               \"B as shipped (in-sample)\": (cached((\"B-shipped\", ()), lambda: fit_B(E, XB, YB, SB)), blocks, False)}\n    # Honest: K-fold by molecule. The shipped np rows are dropped and replaced by our own np rows (same feature\n    # code) from the other folds, so every molecule is scored by rankers that never saw it but did see\n    # in-domain timsTOF rows. Dropping np entirely instead measures domain shift, not generalisation.\n    XA, YA, MA = pv_rows(pv_rank, drop_A)\n    XB, YB, SB, _ = my_rows_(my_rows, [\"np\"] if args.set == \"np\" else [])\n    oA = own_rows(E, runs, [\"base\"], True)\n    oB = own_rows(E, runs, blocks, False)\n    mids_all = np.array(sorted(truth.molecule_id))\n    fold_of = dict(zip(mids_all, np.random.default_rng(0).permutation(len(mids_all)) % args.folds))\n    fA, fB = np.array([fold_of[m] for m in oA[3]]), np.array([fold_of[m] for m in oB[3]])\n    stamp = tuple(os.path.getmtime(cpath(f\"recs_{args.set}_{s}.pkl\")) for s in runs) + (((\"shuffle\", args.shuffle),) if args.shuffle else ()) \\\n        + (((\"np_weight\", args.np_weight),) if args.np_weight != 1 else ())\n    # in-domain up-weighting: our own np rows get np_weight x the per-row weight of the shipped out-of-domain rows\n    wA = lambda a: np.concatenate([np.ones(len(YA)), np.full(a.sum(), args.np_weight)])\n    wB = lambda b: np.concatenate([np.ones(len(YB)), np.full(b.sum(), args.np_weight)])\n    cv = []\n    for f in range(args.folds):\n        a, b = fA != f, fB != f\n        mA = cached((\"A-cv\", args.set, f, args.folds, tuple(sorted(drop_A)), stamp),\n                    lambda: fit_A(E, np.vstack([XA, oA[0][a]]), np.concatenate([YA, oA[1][a]]), np.concatenate([MA, oA[2][a]]), wA(a)))\n        mB = cached((\"B-cv\", args.set, f, args.folds, stamp),\n                    lambda: fit_B(E, np.vstack([XB, oB[0][b]]), np.concatenate([YB, oB[1][b]]), np.concatenate([SB, oB[2][b]]), wB(b)))\n        cv.append((mA, mB))\n        log(f\"fold {f}: rankers ready ({time.time() - t0:.0f}s)\")\n\n    # out-of-fold ranker scores for every molecule of every simulation\n    oof = {}\n    for sim, R in runs.items():\n        oof[sim] = ({}, {})\n        for f, (mA, mB) in enumerate(cv):\n            rf = [r for r in R[\"recs\"] if fold_of[r[\"mid\"]] == f]\n            oof[sim][0].update(E.score_molecules(rf, mA, [\"base\"], True))\n            oof[sim][1].update(E.score_molecules(rf, mB, blocks, False))\n    # Stacker: fitted on the out-of-fold ranker scores of the other folds' molecules (both simulations),\n    # applied to this fold. Its rows are also exported so the notebook fits the same stacker on all of them.\n    st = {\"X\": [], \"Y\": [], \"S\": [], \"mid\": []}\n    for sim, R in runs.items():\n        for r in R[\"recs\"]:\n            if not len(r[\"cand\"]) or r[\"truth_idx\"] < 0: continue\n            X = stack_features(r, oof[sim][0][r[\"mid\"]], oof[sim][1][r[\"mid\"]])\n            y = np.zeros(len(X), np.float32); y[r[\"truth_idx\"]] = 1\n            st[\"X\"].append(X); st[\"Y\"].append(y); st[\"S\"].append(np.full(len(X), 0 if sim == \"c1\" else 1, np.int8))\n            st[\"mid\"].append(np.full(len(X), r[\"mid\"], object))\n    st = {k: (np.vstack(v) if k == \"X\" else np.concatenate(v)) for k, v in st.items()}\n    fS = np.array([fold_of[m] for m in st[\"mid\"]])\n    stackers = [cached((\"stack-cv\", args.set, f, args.folds, stamp, repr(E_STACK.GBM), E_STACK.SEEDS, E_STACK.W_C1),\n                       lambda: fit_stacker(st[\"X\"][fS != f], st[\"Y\"][fS != f], st[\"S\"][fS != f])) for f in range(args.folds)]\n    np.savez_compressed(cpath(f\"stack_rows_{args.set}{sfx}.npz\"), X=st[\"X\"], Y=st[\"Y\"], S=st[\"S\"], mid=st[\"mid\"].astype(str))\n    # our own ranker rows for this set, in the shipped row formats (A: base + M, B: blocks + S), for the notebook\n    np.savez_compressed(cpath(f\"own_rows_{args.set}{sfx}.npz\"), XA=oA[0], YA=oA[1], MA=oA[2], XB=oB[0], YB=oB[1], SB=oB[2],\n                        blocks=np.array(blocks), midA=oA[3].astype(str), midB=oB[3].astype(str))\n    log(f\"stacker: {len(st['Y']):,} rows x {st['X'].shape[1]} features; rows exported\")\n\n    rows = []; rr_all = {}\n    for sim, R in runs.items():\n        recs = R[\"recs\"]; mids = R[\"mids\"]\n        S = channel_scores(E, recs)\n        for name, (models, bl, use_fp) in shipped.items():\n            S[name] = E.score_molecules(recs, models, bl, use_fp)\n        S[\"A cv\"], S[\"B cv\"] = oof[sim]\n        S[\"blend 0.88 cv\"] = E.blend_scores(S[\"A cv\"], S[\"B cv\"], 0.88)\n        S[\"stack cv\"] = {}\n        for f in range(args.folds):\n            rf = [r for r in recs if fold_of[r[\"mid\"]] == f]\n            S[\"stack cv\"].update(score_stacker(rf, stackers[f], S[\"A cv\"], S[\"B cv\"]))\n        S[\"blend 0.88 as shipped (in-sample)\"] = E.blend_scores(S[\"A as shipped (in-sample)\"], S[\"B as shipped (in-sample)\"], 0.88)\n        pickle.dump(S, open(cpath(f\"scores_{args.set}_{sim}{sfx}.pkl\"), \"wb\"), protocol=4)\n        samp = cpath(f\"sample_{args.set}.csv\"); pd.DataFrame({\"molecule_id\": mids}).to_csv(samp, index=False)\n        subs = E.make_submissions([(m, None) for m in mids], recs, S, samp, args.workers)\n        sol = truth.rename(columns={\"normalized_smiles\": \"smiles\"})[[\"molecule_id\", \"smiles\"]]\n        ret = np.mean([r.get(\"truth_idx\", -1) >= 0 for r in recs])\n        ncand = np.median([len(r[\"cand\"]) for r in recs])\n        for name, sub in subs.items():\n            rr = reciprocal_ranks(sub, truth_key, smi_key, metric)\n            v = np.array([rr[m] for m in truth.molecule_id])\n            if name == \"blend 0.88 cv\":\n                official = metric.score(sol, sub, \"molecule_id\")\n                assert abs(official - v.mean()) < 1e-9, (official, v.mean())\n                sub.to_csv(cpath(f\"sub_{args.set}_{sim}{sfx}.csv\"), index=False)\n            rr_all[(sim, name)] = v\n            rows.append(dict(sim=sim, method=name, mrr=v.mean(), se=boot_se(v), top1=(v == 1).mean(),\n                             top25=(v > 0).mean(), retention=ret, median_cand=ncand))\n    df = pd.DataFrame(rows)\n    wide = df.pivot_table(index=\"method\", columns=\"sim\", values=\"mrr\", sort=False)\n    if {\"c1\", \"c2\"} <= set(wide.columns):\n        wide[\"pred_LB\"] = LB_A * wide[\"c1\"] + LB_B * wide[\"c2\"]\n    pd.set_option(\"display.width\", 200); pd.set_option(\"display.max_columns\", 20)\n    print(\"\\n\" + df.round(4).to_string(index=False))\n    print(\"\\n\" + wide.round(4).to_string())\n    if {\"c1\", \"c2\"} <= set(wide.columns):\n        # paired difference vs the blend in predicted-LB units (same molecules in both sims -> resample molecules)\n        base, rng, n = \"blend 0.88 cv\", np.random.default_rng(0), len(truth)\n        for name in (\"lib_shift\", \"stack cv\"):\n            d = LB_A * (rr_all[(\"c1\", name)] - rr_all[(\"c1\", base)]) + LB_B * (rr_all[(\"c2\", name)] - rr_all[(\"c2\", base)])\n            print(f\"paired dLB {name} vs {base}: {d.mean():+.4f} +/- {np.std([d[rng.integers(0, n, n)].mean() for _ in range(2000)]):.4f}\")\n    pickle.dump({f\"{k[0]}|{k[1]}\": v for k, v in rr_all.items()}, open(cpath(f\"rr_{args.set}{sfx}.pkl\"), \"wb\"))\n    out = cpath(f\"report_{args.set}{sfx}.csv\"); df.to_csv(out, index=False)\n    log(f\"report -> {out}\")\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"stage\", choices=[\"build\", \"score\"])\n    ap.add_argument(\"--set\", default=\"np\", choices=sorted(QUERY_SETS))\n    ap.add_argument(\"--sims\", default=\"c1,c2\")\n    ap.add_argument(\"--limit\", type=int, default=0)\n    ap.add_argument(\"--folds\", type=int, default=5)\n    ap.add_argument(\"--shuffle\", type=int, default=1,\n                    help=\"permute candidates within each molecule (seed; 0 = pool order, which leaks -- see score())\")\n    ap.add_argument(\"--np_weight\", type=float, default=1.0,\n                    help=\"sample-weight multiplier on our own in-domain (np) ranker rows in the CV fits\")\n    ap.add_argument(\"--workers\", type=int, default=max(1, (os.cpu_count() or 4) - 2))\n    a = ap.parse_args(); a.sims = a.sims.split(\",\")\n    build(a) if a.stage == \"build\" else score(a)\n", "c3sim.py": "\"\"\"Class-3 simulation, stage 1: can derivative generation reach the answer at all?\n\nClass 3 = the answer is in no database (not in PubChem). The engine scores 0 on it by construction: every\ncandidate comes from the pool. The only route is to generate new structures. Before building a generator\nand a ranker for it, this measures the ceiling on the held-out np molecules.\n\nSetup (reuses the cached Class-2 run, recs_{set}_c2.pkl): all library spectra of the query structure are\nalready removed there, so the spectral analogs in rec[\"ana\"] are exactly what a Class-3 query would get.\nThe answer is then treated as absent from the pool: it can be neither a source nor a candidate.\n\nFor each molecule, apply a fixed set of natural-product transformations (hydroxylation, methylation,\nglycosylation, acylation, redox, ...) to source structures, keep products within PPM_WIN of the query's\nneutral mass, and check whether the answer's tautomer-canonical InChIKey14 (the scorer's key) is among\nthem. Sources:\n  ana@k   the top-k spectral analogs the engine already computes (k = 1, 5, 20, 80)\n  pool    every pool structure at target - delta for some transformation delta (oracle-free but large)\nOne step and two steps (analogs only). Reports reachability, the number of generated candidates per\nmolecule (dilution), and structural distance of the nearest analog.\n\n  python c3sim.py [--set np] [--limit N] [--no-pool] [--two-step 1]\n\"\"\"\nimport os, sys, time, pickle, argparse\nfrom collections import Counter\nHERE = os.path.dirname(os.path.abspath(__file__))\nCACHE = os.path.join(os.path.dirname(HERE), \"data\", \"cache\")\nsys.path.insert(0, HERE)\nimport numpy as np, pandas as pd\n\nPPM = 10.0\nM = {\"C\": 12.0, \"H\": 1.00782503207, \"O\": 15.99491461956, \"N\": 14.0030740048, \"S\": 31.97207100}\n\n\ndef mass(f):\n    return sum(M[k] * v for k, v in f.items())\n\n\n# name, forward reaction SMARTS (source >> product), formula delta. Reverse direction is listed separately.\n# Stereo is ignored throughout: the scorer's key is InChIKey14 (connectivity).\nHEX = \"C1OC(CO)C(O)C(O)C1O\"; DHEX = \"C1OC(C)C(O)C(O)C1O\"; PEN = \"C1OCC(O)C(O)C1O\"; GLA = \"C1OC(C(=O)O)C(O)C(O)C1O\"\nHEX_P = \"[CH1]1-[OX2]-[CH1](-[CH2]-[OH1])-[CH1](-[OH1])-[CH1](-[OH1])-[CH1]1-[OH1]\"\nDHEX_P = \"[CH1]1-[OX2]-[CH1](-[CH3])-[CH1](-[OH1])-[CH1](-[OH1])-[CH1]1-[OH1]\"\nPEN_P = \"[CH1]1-[OX2]-[CH2]-[CH1](-[OH1])-[CH1](-[OH1])-[CH1]1-[OH1]\"\nGLA_P = \"[CH1]1-[OX2]-[CH1](-[CX3](=O)-[OH1])-[CH1](-[OH1])-[CH1](-[OH1])-[CH1]1-[OH1]\"\nACYL = {  # O-acyl esters: name -> (SMILES of acyl group attached at O, SMARTS to remove it, formula delta)\n    \"acetyl\": (\"C(C)=O\", \"[CX3](=O)-[CH3]\", dict(C=2, H=2, O=1)),\n    \"malonyl\": (\"C(=O)CC(=O)O\", \"[CX3](=O)-[CH2]-[CX3](=O)-[OH1]\", dict(C=3, H=2, O=3)),\n    \"coumaroyl\": (\"C(=O)C=Cc1ccc(O)cc1\", \"[CX3](=O)-[CH1]=[CH1]-c1ccc(-[OH1])cc1\", dict(C=9, H=6, O=2)),\n    \"caffeoyl\": (\"C(=O)C=Cc1ccc(O)c(O)c1\", \"[CX3](=O)-[CH1]=[CH1]-c1ccc(-[OH1])c(-[OH1])c1\", dict(C=9, H=6, O=3)),\n    \"feruloyl\": (\"C(=O)C=Cc1ccc(O)c(OC)c1\", \"[CX3](=O)-[CH1]=[CH1]-c1ccc(-[OH1])c(-[OX2]-[CH3])c1\", dict(C=10, H=8, O=3)),\n    \"galloyl\": (\"C(=O)c1cc(O)c(O)c(O)c1\", \"[CX3](=O)-c1cc(-[OH1])c(-[OH1])c(-[OH1])c1\", dict(C=7, H=4, O=4)),\n}\nT = [\n    (\"hydroxyl_aliph\", \"[C;X4;!H0:1]>>[C:1]O\", dict(O=1)),\n    (\"hydroxyl_arom\", \"[c;H1:1]>>[c:1]O\", dict(O=1)),\n    (\"dehydroxyl\", \"[#6;!$([#6]=O):1]-[OX2H1]>>[#6:1]\", dict(O=-1)),\n    (\"O_methyl\", \"[OX2H1:1]>>[O:1]C\", dict(C=1, H=2)),\n    (\"N_methyl\", \"[NX3;H1,H2;!$(N[#6]=O):1]>>[N:1]C\", dict(C=1, H=2)),\n    (\"C_methyl_arom\", \"[c;H1:1]>>[c:1]C\", dict(C=1, H=2)),\n    (\"demethyl\", \"[O,N;X2,X3:1]-[CH3]>>[*:1]\", dict(C=-1, H=-2)),\n    (\"hydrogenate\", \"[C:1]=[C:2]>>[C:1]-[C:2]\", dict(H=2)),\n    (\"dehydrogenate\", \"[C;X4;!H0:1]-[C;X4;!H0:2]>>[C:1]=[C:2]\", dict(H=-2)),\n    (\"keto_reduce\", \"[C;X3;$(C([#6])[#6]):1]=[OX1:2]>>[C:1]-[O:2]\", dict(H=2)),\n    (\"alcohol_oxid\", \"[C;X4;H1:1]-[OX2H1:2]>>[C:1]=[O:2]\", dict(H=-2)),\n    (\"hydrate\", \"[C:1]=[C:2]>>[C:1](O)-[C:2]\", dict(H=2, O=1)),\n    (\"dehydrate\", \"[C;X4;!H0:1]-[C;X4:2]-[OX2H1]>>[C:1]=[C:2]\", dict(H=-2, O=-1)),\n    (\"CH2_to_CO\", \"[C;X4;H2;R:1]>>[C:1]=O\", dict(H=-2, O=1)),\n    (\"CH3_to_COOH\", \"[CH3;$(C[#6]):1]>>[C:1](=O)O\", dict(H=-2, O=2)),\n    (\"sulfate\", \"[OX2H1:1]>>[O:1]S(=O)(=O)O\", dict(S=1, O=3)),\n    (\"desulfate\", \"[OX2:1]-S(=O)(=O)-[OH1]>>[O:1]\", dict(S=-1, O=-3)),\n    (\"prenyl_arom\", \"[c;H1:1]>>[c:1]CC=C(C)C\", dict(C=5, H=8)),\n    (\"prenyl_O\", \"[OX2H1:1]>>[O:1]CC=C(C)C\", dict(C=5, H=8)),\n    (\"deprenyl\", \"[c,O:1]-[CH2]-[CH1]=[CX3](-[CH3])-[CH3]>>[*:1]\", dict(C=-5, H=-8)),\n]\nfor nm, smi, pat, d in ((\"hexose\", HEX, HEX_P, dict(C=6, H=10, O=5)), (\"deoxyhexose\", DHEX, DHEX_P, dict(C=6, H=10, O=4)),\n                        (\"pentose\", PEN, PEN_P, dict(C=5, H=8, O=4)), (\"glucuronide\", GLA, GLA_P, dict(C=6, H=8, O=6))):\n    T.append((f\"+{nm}\", f\"[OX2H1:1]>>[O:1]{smi}\", d))\n    T.append((f\"-{nm}\", f\"[OX2:1]-{pat}>>[O:1]\", {k: -v for k, v in d.items()}))\nfor nm, (smi, pat, d) in ACYL.items():\n    T.append((f\"+{nm}\", f\"[OX2H1:1]>>[O:1]{smi}\", d))\n    T.append((f\"-{nm}\", f\"[OX2:1]-{pat}>>[O:1]\", {k: -v for k, v in d.items()}))\nDELTA = np.array([mass(d) for _, _, d in T])\n\n_w = {}\n\n\ndef _init():\n    from rdkit import RDLogger\n    from rdkit.Chem import AllChem\n    RDLogger.DisableLog(\"rdApp.*\")\n    P = pickle.load(open(os.path.join(CACHE, \"pool.pkl\"), \"rb\"))\n    _w[\"smiles\"], _w[\"mass\"] = P[\"smiles\"], P[\"mass\"]\n    _w[\"rxn\"] = [AllChem.ReactionFromSmarts(s) for _, s, _ in T]\n\n\ndef _key(m):\n    import casmi_engine as E\n    from rdkit import Chem\n    try:\n        return E.canon_key(Chem.MolToSmiles(m))\n    except Exception:\n        return None\n\n\ndef _products(mol, ti):\n    \"\"\"All single-site products of transformation ti on mol, as sanitized mols with formula + plain ik14.\"\"\"\n    from rdkit import Chem\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    out = {}\n    for ps in _w[\"rxn\"][ti].RunReactants((mol,), maxProducts=200):\n        p = ps[0]\n        try:\n            Chem.SanitizeMol(p); smi = Chem.MolToSmiles(p); p = Chem.MolFromSmiles(smi)\n            if p is None: continue\n            k = Chem.MolToInchiKey(p)[:14]\n        except Exception:\n            continue\n        if k and k not in out:\n            out[k] = (p, CalcMolFormula(p))\n    return out\n\n\ndef _mol(i):\n    from rdkit import Chem\n    return Chem.MolFromSmiles(_w[\"smiles\"][int(i)])\n\n\ndef work(job):\n    \"\"\"job: dict(mid, target, truth_key, truth_formula, ana=[pool idx], pool_srcs=bool, two_step=bool)\"\"\"\n    from rdkit.Chem.Descriptors import ExactMolWt\n    t0 = time.time()\n    tgt, tk, tf = job[\"target\"], job[\"truth_key\"], job[\"truth_formula\"]\n    tol = tgt * PPM / 1e6\n    ana = [a for a in job[\"ana\"] if a not in job[\"self_idx\"]]\n    res = dict(mid=job[\"mid\"], n_ana=len(ana))\n    # analog mass deltas vs the transformation table\n    dm = tgt - _w[\"mass\"][np.asarray(ana, np.int64)] if ana else np.zeros(0)\n    res[\"delta_rank1\"] = next((i for i, d in enumerate(dm) if np.abs(DELTA - d).min() <= tol), -1)\n    res[\"same_mass_rank1\"] = next((i for i, d in enumerate(dm) if abs(d) <= tol), -1)\n\n    gen, hit = {}, {}  # plain ik14 -> (source rank or -1 for pool, route); hit route for the answer\n\n    def consider(prods, rank, route):\n        for k, (p, f) in prods.items():\n            if abs(ExactMolWt(p) - tgt) > tol: continue\n            if k not in gen: gen[k] = (rank, route)\n            if f == tf and \"first\" not in hit and _key(p) == tk:\n                hit[\"first\"] = (rank, route)\n    # one step from analogs (delta-matched transformations only)\n    for r, (a, d) in enumerate(zip(ana, dm)):\n        tis = np.where(np.abs(DELTA - d) <= tol)[0]\n        if not len(tis): continue\n        m = _mol(a)\n        if m is None: continue\n        for ti in tis:\n            consider(_products(m, ti), r, T[ti][0])\n    res[\"hit1_rank\"] = hit[\"first\"][0] if \"first\" in hit else -1\n    res[\"hit1_route\"] = hit[\"first\"][1] if \"first\" in hit else \"\"\n    res[\"gen1_ana\"] = len(gen)\n    # two steps from analogs: delta pairs\n    if job[\"two_step\"]:\n        pair = DELTA[:, None] + DELTA[None, :]\n        h2 = dict(hit)\n        hit.clear()\n        for r, (a, d) in enumerate(zip(ana, dm)):\n            ij = np.argwhere(np.abs(pair - d) <= tol)\n            if not len(ij): continue\n            m = _mol(a)\n            if m is None: continue\n            cache = {}\n            for i, j in ij:\n                if i not in cache: cache[i] = _products(m, i)\n                for k1, (p1, _) in list(cache[i].items())[:60]:\n                    consider(_products(p1, j), r, f\"{T[i][0]}>{T[j][0]}\")\n        res[\"hit2_rank\"] = hit[\"first\"][0] if \"first\" in hit else -1\n        res[\"hit2_route\"] = hit[\"first\"][1] if \"first\" in hit else \"\"\n        hit.clear(); hit.update(h2)\n    res[\"gen12_ana\"] = len(gen)\n    # one step from every pool structure at target - delta (capped per transformation)\n    if job[\"pool_srcs\"]:\n        gen_pool = {}\n        hp = {}\n        for ti, d in enumerate(DELTA):\n            lo = np.searchsorted(_w[\"mass\"], tgt - d - tol, \"left\"); hi = np.searchsorted(_w[\"mass\"], tgt - d + tol, \"right\")\n            for s in range(lo, min(hi, lo + 400)):\n                if s in job[\"self_idx\"]: continue\n                m = _mol(s)\n                if m is None: continue\n                for k, (p, f) in _products(m, ti).items():\n                    if abs(ExactMolWt(p) - tgt) > tol: continue\n                    gen_pool.setdefault(k, T[ti][0])\n                    if f == tf and \"first\" not in hp and _key(p) == tk:\n                        hp[\"first\"] = T[ti][0]\n        res[\"hit_pool\"] = hp.get(\"first\", \"\")\n        res[\"gen1_pool\"] = len(gen_pool)\n    res[\"sec\"] = time.time() - t0\n    return res\n\n\ndef tanimoto_nearest(jobs, truth_smiles):\n    \"\"\"Morgan-2 Tanimoto of the answer to its top analogs (how close the spectral neighbourhood is).\"\"\"\n    from rdkit import Chem, DataStructs, RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog(\"rdApp.*\")\n    g = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\n    P = pickle.load(open(os.path.join(CACHE, \"pool.pkl\"), \"rb\"))\n    out = {}\n    for j in jobs:\n        t = Chem.MolFromSmiles(truth_smiles[j[\"mid\"]])\n        ana = [a for a in j[\"ana\"] if a not in j[\"self_idx\"]]\n        fps = [g.GetFingerprint(m) for m in (Chem.MolFromSmiles(P[\"smiles\"][a]) for a in ana) if m is not None]\n        sims = DataStructs.BulkTanimotoSimilarity(g.GetFingerprint(t), fps) if fps and t is not None else [0.0]\n        out[j[\"mid\"]] = (sims[0] if sims else 0.0, max(sims[:5]), max(sims))\n    return out\n\n\ndef main(a):\n    import casmi_engine as E\n    from rdkit import Chem\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    from multiprocessing import Pool as MPool\n    R = pickle.load(open(os.path.join(CACHE, f\"recs_{a.set}_c2.pkl\"), \"rb\"))\n    truth = R[\"truth\"]\n    tsmi = dict(zip(truth.molecule_id, truth.normalized_smiles))\n    tik14 = dict(zip(truth.molecule_id, truth.inchikey14))\n    P = pickle.load(open(os.path.join(CACHE, \"pool.pkl\"), \"rb\"))\n    cm = pickle.load(open(os.path.join(CACHE, \"canon.pkl\"), \"rb\"))  # pool idx -> canonical ik14 (candidates only)\n    jobs = []\n    for r in R[\"recs\"]:\n        if \"target\" not in r: continue\n        tk = E.canon_key(tsmi[r[\"mid\"]])\n        ana = [int(p) for p, _ in r[\"ana\"]]\n        # the answer is absent in Class 3: drop it (and tautomer twins) as a source\n        tik = tik14[r[\"mid\"]]\n        self_idx = {i for i in ana + [int(c) for c in r[\"cand\"]] if cm.get(i) == tk or P[\"keys\"][i] in (tk, tik)}\n        jobs.append(dict(mid=r[\"mid\"], target=float(r[\"target\"]), truth_key=tk,\n                         truth_formula=CalcMolFormula(Chem.MolFromSmiles(tsmi[r[\"mid\"]])),\n                         ana=ana, self_idx=self_idx, pool_srcs=a.pool, two_step=a.two_step))\n    if a.limit: jobs = jobs[:a.limit]\n    t0 = time.time()\n    with MPool(a.workers, initializer=_init) as mp:\n        res = []\n        for i, x in enumerate(mp.imap_unordered(work, jobs, chunksize=1)):\n            res.append(x)\n            if i % 25 == 0: print(f\"[{time.time() - t0:5.0f}s] {i}/{len(jobs)}\", flush=True)\n    df = pd.DataFrame(res).set_index(\"mid\")\n    tan = tanimoto_nearest(jobs, tsmi)\n    df[\"tan_top1\"], df[\"tan_top5\"], df[\"tan_max\"] = zip(*[tan[m] for m in df.index])\n    df.to_csv(os.path.join(CACHE, f\"c3_reach_{a.set}.csv\"))\n\n    n = len(df)\n    print(f\"\\nClass-3 reachability on {a.set}: {n} molecules, {len(T)} transformations, {PPM:g} ppm\")\n    print(f\"answer found among spectral analogs? no (removed); nearest-analog Morgan2 Tanimoto \"\n          f\"top1 median {df.tan_top1.median():.2f}, best-of-80 median {df.tan_max.median():.2f}, \"\n          f\"best-of-80 >= 0.7: {(df.tan_max >= 0.7).mean():.1%}\")\n    print(f\"same-mass analog (isomer) within top 1/5/80: \" + \" / \".join(\n        f\"{((df.same_mass_rank1 >= 0) & (df.same_mass_rank1 < k)).mean():.1%}\" for k in (1, 5, 80)))\n    print(f\"mass delta of some analog matches a transformation, top 1/5/20/80: \" + \" / \".join(\n        f\"{((df.delta_rank1 >= 0) & (df.delta_rank1 < k)).mean():.1%}\" for k in (1, 5, 20, 80)))\n    for col, lab in ((\"hit1_rank\", \"1 step\"), (\"hit2_rank\", \"2 steps\")):\n        if col in df:\n            print(f\"answer GENERATED ({lab}) from analogs top 1/5/20/80: \" + \" / \".join(\n                f\"{((df[col] >= 0) & (df[col] < k)).mean():.1%}\" for k in (1, 5, 20, 80)))\n    any_ana = (df.hit1_rank >= 0) | (df.get(\"hit2_rank\", pd.Series(-1, df.index)) >= 0)\n    print(f\"answer generated from analogs, 1 or 2 steps: {any_ana.mean():.1%}\")\n    if \"hit_pool\" in df:\n        print(f\"answer GENERATED (1 step) from any pool structure: {(df.hit_pool != '').mean():.1%}; \"\n              f\"union with analogs {(any_ana | (df.hit_pool != '')).mean():.1%}\")\n    print(f\"generated candidates in window per molecule, median: analogs 1-step {df.gen1_ana.median():.0f}, \"\n          f\"1+2-step {df.gen12_ana.median():.0f}\" + (f\", pool 1-step {df.gen1_pool.median():.0f}\" if \"gen1_pool\" in df else \"\"))\n    routes = Counter(df.hit1_route[df.hit1_route != \"\"])\n    if \"hit2_route\" in df: routes.update(df.hit2_route[(df.hit1_route == \"\") & (df.hit2_route != \"\")])\n    print(\"routes that reach the answer:\", dict(routes.most_common(15)))\n    if \"hit_pool\" in df: print(\"pool routes:\", dict(Counter(df.hit_pool[df.hit_pool != \"\"]).most_common(10)))\n    print(f\"median seconds/molecule {df.sec.median():.1f} (max {df.sec.max():.0f})\")\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--set\", default=\"np\")\n    ap.add_argument(\"--limit\", type=int, default=0)\n    ap.add_argument(\"--pool\", type=int, default=1)\n    ap.add_argument(\"--two_step\", type=int, default=1)\n    ap.add_argument(\"--workers\", type=int, default=max(1, (os.cpu_count() or 4) - 2))\n    main(ap.parse_args())\n", "stage2.py": "\"\"\"Class-3 stage 2: one-step analog products as extra candidates, measured on three simulations.\n\nc3sim.py showed the answer of a Class-3 query (absent from every database) is a single NP transformation away\nfrom one of the engine's top-80 spectral analogs for ~39% of np molecules. Here those products become candidates\nwith the engine's own features, and the net leaderboard effect is measured:\n\n  c1  Class-1 simulation + generated candidates (dilution cost)\n  c2  Class-2 simulation + generated candidates (dilution cost)\n  c3  Class-3 simulation: the c2 run with the answer (and tautomer twins) removed from the pool and from the analog\n      sources; the answer can only come back as a generated product.\n\nGenerated candidates get every per-candidate feature the pool candidates get -- fingerprint (FPNet f.z, analog\nTanimoto), MetFrag-lite scores, mass -- plus a `gen` block: generated flag, best source-analog spectral similarity,\n1/(1+best source rank), number of distinct source analogs. Library features are 0 (no reference spectra exist).\nA product that duplicates a candidate already in the window (tautomer-canonical InChIKey14) is dropped.\n\n  python stage2.py gen   [--set np]                  # products + features -> cache/gen_{set}.pkl\n  python stage2.py score [--set np] [--c3_weight 1]   # rankers, three sims, report\n\nMethods in the report:\n  base            the honest harness baseline (cached CV rankers, pool candidates only; c3 = 0 by construction)\n  base+gen        the same rankers applied to pool + generated candidates (rankers never saw generated rows)\n  gen A/B/blend   CV rankers refit with the gen block and our own c1/c2/c3 rows (c3 rows x --c3_weight)\n\"\"\"\nimport os, sys, time, pickle, argparse, copy, hashlib\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport harness as H  # sets CASMI_ROOTS\nfrom harness import cpath, log\nimport c3sim as C3\nimport numpy as np, pandas as pd\n\nGEN_COLS = (\"gen\", \"gen_sim\", \"gen_rinv\", \"gen_nsrc\")\nLIB_COLS = (\"lib\", \"lib_mean\", \"libsh\", \"lib_cnt\")\nFRAG_COLS = (\"frag\", \"frag_ad\", \"frag_mean\", \"frag_pol\")\nPER_CAND = H.PER_CAND + GEN_COLS\n# Class-3 value in LB units: everything that is neither Class-1 (0.162) nor Class-2 x pool recall (0.271).\n# An upper bound: np answers are known COCONUT compounds whose relatives stay in the pool (see c3sim.py).\nLB_C = 1.0 - H.LB_A - H.LB_B\n\n\n# ---------------------------------------------------------------------------------------------\n# gen: products per molecule\n# ---------------------------------------------------------------------------------------------\ndef _init():\n    from rdkit import RDLogger\n    from rdkit.Chem import AllChem\n    import casmi_engine as E\n    RDLogger.DisableLog(\"rdApp.*\")\n    C3._w[\"rxn\"] = [AllChem.ReactionFromSmarts(s) for _, s, _ in C3.T]\n    # fp_and_mass only initialises its generators while casmi_engine._g is empty; a forked (Linux) worker inherits\n    # the parent's _g, already holding the tautomer enumerator from canon_key, and would silently return None\n    E._fp_init()\n\n\ndef work(job):\n    \"\"\"job: sim, mid, target, src = [(smiles, mass, rank, spectral sim)]. All one-step products within PPM of the\n    target, aggregated per plain InChIKey14 (best source rank / similarity, distinct sources).\"\"\"\n    from rdkit import Chem\n    from rdkit.Chem.Descriptors import ExactMolWt\n    import casmi_engine as E, pv\n    tgt = job[\"target\"]; tol = tgt * C3.PPM / 1e6\n    agg = {}\n    for smi, m0, rank, s in job[\"src\"]:\n        tis = np.where(np.abs(C3.DELTA - (tgt - m0)) <= tol)[0]\n        if not len(tis): continue\n        mol = Chem.MolFromSmiles(smi)\n        if mol is None: continue\n        for ti in tis:\n            for k, (p, _) in C3._products(mol, ti).items():\n                if abs(ExactMolWt(p) - tgt) > tol: continue\n                a = agg.get(k)\n                if a is None:\n                    agg[k] = a = dict(smiles=Chem.MolToSmiles(p), rank=rank, sim=s, src=set(), route=C3.T[ti][0])\n                a[\"rank\"] = min(a[\"rank\"], rank); a[\"sim\"] = max(a[\"sim\"], s); a[\"src\"].add(rank)\n    out = []\n    for k, a in agg.items():\n        fm = E.fp_and_mass(a[\"smiles\"])\n        ck = E.canon_key(a[\"smiles\"])\n        if fm is None or ck is None: continue\n        out.append(dict(ik=k, ckey=ck, smiles=a[\"smiles\"], fp=fm[0], mass=fm[1], rank=a[\"rank\"], sim=a[\"sim\"],\n                        nsrc=len(a[\"src\"]), route=a[\"route\"], fm=pv.frag_masses_safe(a[\"smiles\"])))\n    return job[\"sim\"], job[\"mid\"], out\n\n\ndef query_specs(set_name):\n    \"\"\"Per-molecule spectra exactly as compute_channels builds them (compute_frag pops them from the cached recs).\"\"\"\n    te, _ = H.build_queries(set_name)\n    out = {}\n    for mid, sub in te.groupby(\"molecule_id\", sort=True):\n        specs = []\n        for r in sub.itertuples():\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\n            k_ = a_ <= float(r.precursor_mz) + 2.0\n            a_, b_ = a_[k_], b_[k_]\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\n        out[mid] = specs\n    return out\n\n\ndef frag_feats(r, specs, fms):\n    \"\"\"compute_frag's four MetFrag-lite scores for a list of fragment-mass arrays.\"\"\"\n    import pv\n    from pv import CFG\n    peaks = []\n    for a_, b_ in specs:\n        m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\n        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n    F1 = np.zeros((len(fms), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\n    for j, fm in enumerate(fms):\n        for k, (x, y) in enumerate(peaks):\n            F1[j, k] = pv.explain_score(fm, x, y, mode=r[\"mode\"], tol=CFG.MZ_TOL)\n            F2[j, k] = pv.explain_score_adduct(fm, x, y, r[\"adducts\"][k], tol=CFG.MZ_TOL)\n            F3[j, k] = pv.explain_score(fm, x, y, mode=r[\"modes\"][k], tol=CFG.MZ_TOL)\n    return dict(frag=F1.max(1), frag_ad=F2.max(1), frag_mean=F3.mean(1), frag_pol=F3.max(1))\n\n\ndef self_indices(r, P, cm, tk, tik):\n    \"\"\"Pool entries that are the answer under the scorer's key (or its plain ik14): absent in Class 3.\"\"\"\n    return {i for i in [int(p) for p, _ in r[\"ana\"]] + [int(c) for c in r[\"cand\"]]\n            if cm.get(i) == tk or P[\"keys\"][i] in (tk, tik)}\n\n\ndef finalize(r, prods, specs, cm, P, drop=()):\n    \"\"\"One molecule's products -> the generated-candidate block: drop duplicates of window candidates (tautomer-canonical\n    ik14 or plain ik14; `drop` = pool entries removed from the window) and of each other, then MetFrag-lite features.\"\"\"\n    cand = [int(c) for c in r[\"cand\"] if int(c) not in drop]\n    have = {cm.get(c) for c in cand} | {P[\"keys\"][c] for c in cand}\n    seen, keep = set(), []\n    for p in sorted(prods, key=lambda p: (p[\"rank\"], -p[\"sim\"], p[\"ckey\"])):\n        if p[\"ckey\"] in have or p[\"ik\"] in have or p[\"ckey\"] in seen: continue\n        seen.add(p[\"ckey\"]); keep.append(p)\n    g = dict(smiles=[p[\"smiles\"] for p in keep], ckey=[p[\"ckey\"] for p in keep], ik=[p[\"ik\"] for p in keep],\n             mass=np.array([p[\"mass\"] for p in keep]), route=[p[\"route\"] for p in keep],\n             fp=np.stack([p[\"fp\"] for p in keep]) if keep else np.zeros((0, 867), np.uint8),\n             gen_sim=np.array([p[\"sim\"] for p in keep], np.float32),\n             gen_rinv=np.array([1.0 / (1 + p[\"rank\"]) for p in keep], np.float32),\n             gen_nsrc=np.array([p[\"nsrc\"] for p in keep], np.float32))\n    g.update(frag_feats(r, specs, [p[\"fm\"] for p in keep]) if keep else {k: np.zeros(0, np.float32) for k in FRAG_COLS})\n    return g\n\n\ndef run_products(jobs, workers):\n    from multiprocessing import Pool as MPool\n    t0 = time.time(); res = {}\n    with MPool(workers, initializer=_init) as mp:\n        for i, (sim, mid, out) in enumerate(mp.imap_unordered(work, jobs, chunksize=1)):\n            res[(sim, mid)] = out\n            if i % 100 == 0: log(f\"  products {i}/{len(jobs)} ({time.time() - t0:.0f}s)\")\n    return res\n\n\ndef gen(a):\n    import casmi_engine as E, metric, pv\n    metric.install_dependencies()\n    P = pickle.load(open(cpath(\"pool.pkl\"), \"rb\"))\n    runs = {s: pickle.load(open(cpath(f\"recs_{a.set}_{s}.pkl\"), \"rb\")) for s in (\"c1\", \"c2\")}\n    E.POOL = P  # canon_map reads POOL[\"smiles\"]; builds cache/canon.pkl on a fresh machine\n    cm = H.canon_map(E, np.concatenate([r[\"cand\"] for R in runs.values() for r in R[\"recs\"]]), a.workers)\n    truth = runs[\"c2\"][\"truth\"]\n    tsmi, tik14 = dict(zip(truth.molecule_id, truth.normalized_smiles)), dict(zip(truth.molecule_id, truth.inchikey14))\n    tkey = {m: metric.canonicalize_smiles_to_ik14(s) for m, s in tsmi.items()}\n    agree = np.mean([E.canon_key(tsmi[m]) == k for m, k in tkey.items()])\n    log(f\"truth keys: engine canon_key agrees with the metric on {agree:.1%}\")\n    jobs, drop = [], {}\n    for sim in (\"c1\", \"c2\", \"c3\"):\n        for r in runs[\"c1\" if sim == \"c1\" else \"c2\"][\"recs\"]:\n            if \"target\" not in r: continue\n            ana = [(int(p), float(s)) for p, s in r[\"ana\"]]\n            if sim == \"c3\":\n                si = self_indices(r, P, cm, tkey[r[\"mid\"]], tik14[r[\"mid\"]])\n                drop[r[\"mid\"]] = si\n                ana = [(p, s) for p, s in ana if p not in si]\n            src = [(P[\"smiles\"][p], float(P[\"mass\"][p]), rank, s) for rank, (p, s) in enumerate(ana)]\n            jobs.append(dict(sim=sim, mid=r[\"mid\"], target=float(r[\"target\"]), src=src))\n    del P[\"k2i\"]\n    res = run_products(jobs, a.workers)\n    specs = query_specs(a.set)\n    # the rebuilt spectra must reproduce the cached MetFrag scores of pool candidates\n    chk = []\n    for r in runs[\"c2\"][\"recs\"][:20]:\n        if not len(r[\"cand\"]): continue\n        f = frag_feats(r, specs[r[\"mid\"]], [pv.frag_masses_safe(P[\"smiles\"][int(c)]) for c in r[\"cand\"][:5]])\n        chk.append(max(np.abs(f[k] - np.asarray(r[k])[:5]).max() for k in FRAG_COLS))\n    log(f\"MetFrag reproduction on cached pool candidates: max abs diff {max(chk):.2e}\")\n    assert max(chk) < 1e-5\n    G = {s: {} for s in (\"c1\", \"c2\", \"c3\")}; stats = []\n    for sim in G:\n        for r in runs[\"c1\" if sim == \"c1\" else \"c2\"][\"recs\"]:\n            if (sim, r[\"mid\"]) not in res: continue\n            g = finalize(r, res[(sim, r[\"mid\"])], specs[r[\"mid\"]], cm, P, drop[r[\"mid\"]] if sim == \"c3\" else ())\n            if sim == \"c3\": g[\"drop\"] = sorted(drop[r[\"mid\"]])\n            G[sim][r[\"mid\"]] = g\n            hit = tkey[r[\"mid\"]] in g[\"ckey\"]\n            stats.append(dict(sim=sim, mid=r[\"mid\"], n_raw=len(res[(sim, r[\"mid\"])]), n_gen=len(g[\"smiles\"]), answer_generated=hit,\n                              answer_route=g[\"route\"][g[\"ckey\"].index(tkey[r[\"mid\"]])] if hit else \"\"))\n    pickle.dump(dict(G=G, truth_key=tkey), open(cpath(f\"gen_{a.set}.pkl\"), \"wb\"), protocol=4)\n    st = pd.DataFrame(stats); st.to_csv(cpath(f\"gen_stats_{a.set}.csv\"), index=False)\n    assert st.n_gen.sum() > 0 and st[st.sim == \"c3\"].answer_generated.any(), \"no products generated\"\n    for sim, d in st.groupby(\"sim\"):\n        log(f\"[{sim}] generated per molecule (after dedup vs window): median {d.n_gen.median():.0f}, mean {d.n_gen.mean():.1f}, \"\n            f\"max {d.n_gen.max()}; raw median {d.n_raw.median():.0f}; answer among generated {d.answer_generated.mean():.1%}\")\n    log(f\"gen -> {cpath(f'gen_{a.set}.pkl')}\")\n\n\n# ---------------------------------------------------------------------------------------------\n# score\n# ---------------------------------------------------------------------------------------------\ndef blk_gen(r, cfp):\n    n = len(r[\"cand\"])\n    return np.column_stack([np.asarray(r[k], np.float32) if k in r else np.zeros(n, np.float32) for k in GEN_COLS])\n\n\ndef permute(recs, rng):\n    for r in recs:\n        p = rng.permutation(len(r[\"cand\"]))\n        for k in PER_CAND:\n            if k in r: r[k] = np.asarray(r[k])[p]\n\n\ndef own_rows3(E, runs, blocks, use_fp):\n    \"\"\"Our ranker rows from c1/c2/c3. M: 0 = c1, 1 = c2, 2 = c3. c3 molecules whose answer was not generated stay in\n    (all-negative groups): they carry the base rate at which a generated candidate is wrong.\"\"\"\n    code = {\"c1\": 0, \"c2\": 1, \"c3\": 2}\n    Xs, Ys, Ms, Ids = [], [], [], []\n    for sim, recs in runs.items():\n        for r in recs:\n            if not len(r[\"cand\"]) or (r[\"truth_idx\"] < 0 and sim != \"c3\"): continue\n            rr = r if use_fp else dict(r, zlog=None)\n            cfp = E.pool_fps(r[\"cand\"])\n            X = np.hstack([E.BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\n            y = np.zeros(len(X), np.float32)\n            if r[\"truth_idx\"] >= 0: y[r[\"truth_idx\"]] = 1\n            Xs.append(X); Ys.append(y); Ms.append(np.full(len(X), code[sim])); Ids.append([r[\"mid\"]] * len(X))\n    return np.vstack(Xs), np.concatenate(Ys), np.concatenate(Ms), np.concatenate(Ids)\n\n\ndef score(a):\n    import casmi_engine as E, metric\n    metric.install_dependencies()\n    if a.smoke:  # tiny GBMs: a different cache key, so every fit path runs end to end in minutes (numbers meaningless)\n        E.CFG.GBM = dict(E.CFG.GBM, max_iter=3); E.RANK.GBM = dict(E.RANK.GBM, max_iter=3)\n    P = pickle.load(open(cpath(\"pool.pkl\"), \"rb\")); pool_fp = np.load(cpath(\"pool_fp.npy\"), mmap_mode=\"r\")\n    N = len(P[\"mass\"]); E.POOL = dict(P, fp=pool_fp)\n    runs, truth, truth_key, base, G, cm = np_sets(E, metric, a)\n    gidx = extend_pool(E, P, pool_fp, G, cm)\n    ext = {s: extend_recs(base[s], G[s], gidx) for s in base}\n    rng2 = np.random.default_rng(a.shuffle + 1000)  # generated candidates were appended at the end: shuffle again\n    for s in (\"c1\", \"c2\", \"c3\"): permute(ext[s], rng2)\n    set_truth(list(base.values()) + list(ext.values()), cm, truth_key)\n    for s in ext:\n        log(f\"[{s}] answer in candidates: pool-only {np.mean([r['truth_idx'] >= 0 for r in base[s]]):.1%}, \"\n            f\"+generated {np.mean([r['truth_idx'] >= 0 for r in ext[s]]):.1%}; median candidates \"\n            f\"{np.median([len(r['cand']) for r in base[s]]):.0f} -> {np.median([len(r['cand']) for r in ext[s]]):.0f}\")\n    _score_rest(E, metric, a, P, N, runs, truth, truth_key, base, ext, cm, gidx)\n\n\ndef set_truth(rec_lists, cm, truth_key):\n    for R in rec_lists:\n        for r in R:\n            hit = [i for i, c in enumerate(r[\"cand\"]) if cm.get(int(c)) == truth_key[r[\"mid\"]]]\n            r[\"truth_idx\"] = hit[0] if hit else -1\n\n\ndef np_sets(E, metric, a):\n    \"\"\"The cached simulations as ranker/evaluation sets: c1, c2 and c3 (= c2 minus the answer), candidates shuffled with\n    the harness' permutation (`harness.py score --shuffle 1` for c1/c2, then c3). E.POOL must be the plain pool.\"\"\"\n    G = pickle.load(open(cpath(f\"gen_{a.set}.pkl\"), \"rb\"))[\"G\"]\n    runs = {s: pickle.load(open(cpath(f\"recs_{a.set}_{s}.pkl\"), \"rb\")) for s in (\"c1\", \"c2\")}\n    truth = runs[\"c2\"][\"truth\"]\n    truth_key = {m: metric.canonicalize_smiles_to_ik14(s) for m, s in zip(truth.molecule_id, truth.normalized_smiles)}\n    base = {s: R[\"recs\"] for s, R in runs.items()}\n    # c3 = c2 with the answer removed from the window (before any shuffling)\n    c3 = []\n    for r in base[\"c2\"]:\n        r = dict(r)\n        if len(r[\"cand\"]) and r[\"mid\"] in G[\"c3\"]:\n            keep = ~np.isin(r[\"cand\"], G[\"c3\"][r[\"mid\"]][\"drop\"])\n            for k in PER_CAND:\n                if k in r: r[k] = np.asarray(r[k])[keep]\n        c3.append(r)\n    base[\"c3\"] = c3\n    cm = H.canon_map(E, np.concatenate([r[\"cand\"] for R in base.values() for r in R]), a.workers)\n    # identical permutation to `harness.py score --shuffle 1` for c1 and c2, then c3\n    rng = np.random.default_rng(a.shuffle)\n    for s in (\"c1\", \"c2\", \"c3\"): permute(base[s], rng)\n    return runs, truth, truth_key, base, G, cm\n\n\ndef extend_pool(E, P, pool_fp, G, cm):\n    \"\"\"Generated structures appended after the N pool entries (one global table over every set in G); E.POOL,\n    E.pool_fps and the gen feature block switch to the extended pool. cm gains their keys. Returns smiles -> index.\"\"\"\n    N = len(P[\"mass\"])\n    gsmi, gidx, gfp, gmass, gkeys = [], {}, [], [], []\n    for sim in G:\n        for g in G[sim].values():\n            for j, s in enumerate(g[\"smiles\"]):\n                if s not in gidx:\n                    gidx[s] = N + len(gsmi); gsmi.append(s); gfp.append(g[\"fp\"][j]); gmass.append(g[\"mass\"][j]); gkeys.append(g[\"ik\"][j])\n                    cm_g = g[\"ckey\"][j]; cm[gidx[s]] = cm_g\n    gfp = np.stack(gfp)\n    E.POOL = dict(fp=pool_fp, mass=np.concatenate([P[\"mass\"], gmass]), smiles=np.concatenate([P[\"smiles\"], np.array(gsmi, object)]),\n                  keys=np.concatenate([P[\"keys\"], np.array(gkeys, object)]))\n\n    def pool_fps(idx):\n        idx = np.asarray(idx, np.int64); out = np.empty((len(idx), pool_fp.shape[1]), np.uint8)\n        m = idx < N\n        out[m] = pool_fp[idx[m]]; out[~m] = gfp[idx[~m] - N]\n        return np.unpackbits(out, axis=1)[:, :6930]\n    E.pool_fps = pool_fps\n    E.BLOCKS[\"gen\"] = blk_gen\n    log(f\"extended pool: {N:,} + {len(gsmi):,} generated structures\")\n    return gidx\n\n\ndef extend_recs(recs, Gs, gidx):\n    \"\"\"Copies of recs with the generated block appended (library features 0: no reference spectra exist).\"\"\"\n    out = []\n    for r in recs:\n        r = dict(r); g = Gs.get(r[\"mid\"])\n        if g is not None and len(r[\"cand\"]) and len(g[\"smiles\"]):\n            n0, n1 = len(r[\"cand\"]), len(g[\"smiles\"])\n            r[\"cand\"] = np.concatenate([r[\"cand\"], [gidx[s] for s in g[\"smiles\"]]]).astype(np.int64)\n            for k in LIB_COLS: r[k] = np.concatenate([np.asarray(r[k], np.float32), np.zeros(n1, np.float32)])\n            for k in FRAG_COLS: r[k] = np.concatenate([np.asarray(r[k], np.float32), g[k]])\n            r[\"gen\"] = np.concatenate([np.zeros(n0, np.float32), np.ones(n1, np.float32)])\n            for k in GEN_COLS[1:]: r[k] = np.concatenate([np.zeros(n0, np.float32), g[k]])\n        out.append(r)\n    return out\n\n\ndef ranker_rows(E, a, base, truth):\n    \"\"\"Shipped row files with the np set held out (replaced by our own np rows): Ranker A (prvsiyan) and B (megayak).\"\"\"\n    pv_rank = E.find(\"rank_train.npz\"); my_rows = E.find(\"sim_rank_rows_nofp.npz\")\n    g_, d_ = H.np_groups_in_pv_rows(E, base[\"c2\"], pv_rank)\n    drop_A = set(g_[d_ < 1e-2].tolist())\n    if a.set == \"np\" and drop_A and max(drop_A) < len(truth): drop_A = set(range(len(truth)))\n    XA, YA, MA = H.pv_rows(pv_rank, drop_A)\n    XB, YB, SB, blocks = H.my_rows_(my_rows, [\"np\"] if a.set == \"np\" else [])\n    return drop_A, (XA, YA, MA), (XB, YB, SB), blocks\n\n\ndef _score_rest(E, metric, a, P, N, runs, truth, truth_key, base, ext, cm, gidx):\n    # rankers ----------------------------------------------------------------------------------\n    drop_A, (XA, YA, MA), (XB, YB, SB), blocks = ranker_rows(E, a, base, truth)\n\n    def cached(key, fn):\n        h = hashlib.md5(repr((key, E.CFG.GBM, E.RANK.GBM, E.RANK.W_A, E.RANK.SEEDS)).encode()).hexdigest()[:12]\n        f = cpath(f\"rankers_{h}.pkl\")\n        if os.path.exists(f): return pickle.load(open(f, \"rb\")), True\n        v = fn(); pickle.dump(v, open(f, \"wb\"), protocol=4); return v, False\n\n    mids_all = np.array(sorted(truth.molecule_id))\n    fold_of = dict(zip(mids_all, np.random.default_rng(0).permutation(len(mids_all)) % a.folds))\n    # baseline rankers: the very models `harness.py score` cached (same key), rows of c1/c2 pool candidates only\n    stamp = tuple(os.path.getmtime(cpath(f\"recs_{a.set}_{s}.pkl\")) for s in (\"c1\", \"c2\")) + ((\"shuffle\", a.shuffle),)\n    lazy = {}\n\n    def base_rows():\n        if not lazy:\n            b2 = {s: {\"recs\": base[s]} for s in (\"c1\", \"c2\")}\n            lazy[\"A\"] = H.own_rows(E, b2, [\"base\"], True); lazy[\"B\"] = H.own_rows(E, b2, blocks, False)\n        return lazy[\"A\"], lazy[\"B\"]\n    t0 = time.time(); cv_base, cv_gen = [], []\n    for f in range(a.folds):\n        def fitA():\n            oA = base_rows()[0]; m = np.array([fold_of[x] for x in oA[3]]) != f\n            return H.fit_A(E, np.vstack([XA, oA[0][m]]), np.concatenate([YA, oA[1][m]]), np.concatenate([MA, oA[2][m]]), np.ones(len(YA) + m.sum()))\n        def fitB():\n            oB = base_rows()[1]; m = np.array([fold_of[x] for x in oB[3]]) != f\n            return H.fit_B(E, np.vstack([XB, oB[0][m]]), np.concatenate([YB, oB[1][m]]), np.concatenate([SB, oB[2][m]]), np.ones(len(YB) + m.sum()))\n        mA, hA = cached((\"A-cv\", a.set, f, a.folds, tuple(sorted(drop_A)), stamp), fitA)\n        mB, hB = cached((\"B-cv\", a.set, f, a.folds, stamp), fitB)\n        cv_base.append((mA, mB))\n        log(f\"fold {f}: baseline rankers {'cached' if hA and hB else 'REFIT'} ({time.time() - t0:.0f}s)\")\n\n    oA = own_rows3(E, ext, [\"base\", \"gen\"], True)\n    oB = own_rows3(E, ext, blocks + [\"gen\"], False)\n    XAg = np.hstack([XA, np.zeros((len(XA), len(GEN_COLS)), np.float32)])\n    XBg = np.hstack([XB, np.zeros((len(XB), len(GEN_COLS)), np.float32)])\n    fA, fB = np.array([fold_of[m] for m in oA[3]]), np.array([fold_of[m] for m in oB[3]])\n    wA = lambda m: np.concatenate([np.ones(len(YA)), np.where(oA[2][m] == 2, a.c3_weight, 1.0)])\n    wB = lambda m: np.concatenate([np.ones(len(YB)), np.where(oB[2][m] == 2, a.c3_weight, 1.0)])\n    gstamp = stamp + (os.path.getmtime(cpath(f\"gen_{a.set}.pkl\")), (\"c3_weight\", a.c3_weight), \"stage2-v1\")\n    log(f\"gen rankers: own rows A {len(oA[1]):,} (generated {int(oA[0][:, -4].sum()):,}, c3 positives \"\n        f\"{int(oA[1][oA[2] == 2].sum())}) + shipped {len(YA):,}; B own {len(oB[1]):,} + shipped {len(YB):,}\")\n    for f in range(a.folds):\n        a_, b_ = fA != f, fB != f\n        mA, _ = cached((\"A-gen-cv\", a.set, f, a.folds, tuple(sorted(drop_A)), gstamp),\n                       lambda: H.fit_A(E, np.vstack([XAg, oA[0][a_]]), np.concatenate([YA, oA[1][a_]]), np.concatenate([MA, oA[2][a_]]), wA(a_)))\n        mB, _ = cached((\"B-gen-cv\", a.set, f, a.folds, gstamp),\n                       lambda: H.fit_B(E, np.vstack([XBg, oB[0][b_]]), np.concatenate([YB, oB[1][b_]]), np.concatenate([SB, oB[2][b_]]), wB(b_)))\n        cv_gen.append((mA, mB))\n        log(f\"fold {f}: gen rankers ready ({time.time() - t0:.0f}s)\")\n\n    # scores, submissions, metric --------------------------------------------------------------\n    def oof(recs, models, bl, use_fp, k):\n        out = {}\n        for f in range(a.folds):\n            out.update(E.score_molecules([r for r in recs if fold_of[r[\"mid\"]] == f], models[f][k], bl, use_fp))\n        return out\n    smi_key = {E.POOL[\"smiles\"][i]: k for i, k in cm.items()}\n    sol = truth.rename(columns={\"normalized_smiles\": \"smiles\"})[[\"molecule_id\", \"smiles\"]]\n    rows, rr_all = [], {}\n    samp = cpath(f\"sample_{a.set}.csv\")\n    for sim in (\"c1\", \"c2\", \"c3\"):\n        mids = runs[\"c2\"][\"mids\"]\n        pd.DataFrame({\"molecule_id\": mids}).to_csv(samp, index=False)\n        S = {\"A\": oof(base[sim], cv_base, [\"base\"], True, 0), \"B\": oof(base[sim], cv_base, blocks, False, 1)}\n        sets = [(base[sim], {\"base\": E.blend_scores(S[\"A\"], S[\"B\"], 0.88)})]\n        S = {\"A\": oof(ext[sim], cv_base, [\"base\"], True, 0), \"B\": oof(ext[sim], cv_base, blocks, False, 1)}\n        sx = {\"base+gen\": E.blend_scores(S[\"A\"], S[\"B\"], 0.88)}\n        S = {\"A\": oof(ext[sim], cv_gen, [\"base\", \"gen\"], True, 0), \"B\": oof(ext[sim], cv_gen, blocks + [\"gen\"], False, 1)}\n        sx.update({\"gen A\": S[\"A\"], \"gen B\": S[\"B\"], \"gen blend 0.88\": E.blend_scores(S[\"A\"], S[\"B\"], 0.88)})\n        sets.append((ext[sim], sx))\n        for recs, ss in sets:\n            subs = E.make_submissions([(m, None) for m in mids], recs, ss, samp, a.workers)\n            ret = np.mean([r[\"truth_idx\"] >= 0 for r in recs]); ncand = np.median([len(r[\"cand\"]) for r in recs])\n            for name, sub in subs.items():\n                rr = H.reciprocal_ranks(sub, truth_key, smi_key, metric)\n                v = np.array([rr[m] for m in truth.molecule_id])\n                if name in (\"base\", \"gen blend 0.88\"):\n                    official = metric.score(sol, sub, \"molecule_id\")\n                    assert abs(official - v.mean()) < 1e-9, (official, v.mean())\n                rr_all[(sim, name)] = v\n                by = {r[\"mid\"]: r for r in recs}\n                gshare = np.mean([(by[m][\"cand\"][np.argsort(-sc, kind=\"mergesort\")][:25] >= N).mean() for m, sc in ss[name].items()])\n                rows.append(dict(sim=sim, method=name, mrr=v.mean(), se=H.boot_se(v), top1=(v == 1).mean(), top25=(v > 0).mean(),\n                                 retention=ret, median_cand=ncand, gen_share_top25=gshare))\n        log(f\"[{sim}] scored\")\n    df = pd.DataFrame(rows)\n    wide = df.pivot_table(index=\"method\", columns=\"sim\", values=\"mrr\", sort=False)\n    wide[\"pred_LB c1c2\"] = H.LB_A * wide[\"c1\"] + H.LB_B * wide[\"c2\"]\n    wide[\"pred_LB +c3\"] = wide[\"pred_LB c1c2\"] + LB_C * wide[\"c3\"]\n    pd.set_option(\"display.width\", 200); pd.set_option(\"display.max_columns\", 20)\n    print(\"\\n\" + df.round(4).to_string(index=False))\n    print(f\"\\npred LB = {H.LB_A} c1 + {H.LB_B} c2 (+ {LB_C:.3f} c3, the residual share; np c3 is an upper bound)\")\n    print(wide.round(4).to_string())\n    n, rng3 = len(truth), np.random.default_rng(0)\n    for name in (\"base+gen\", \"gen blend 0.88\"):\n        for lab, wc in ((\"c1c2\", 0.0), (\"+c3\", LB_C)):\n            d = H.LB_A * (rr_all[(\"c1\", name)] - rr_all[(\"c1\", \"base\")]) + H.LB_B * (rr_all[(\"c2\", name)] - rr_all[(\"c2\", \"base\")]) \\\n                + wc * (rr_all[(\"c3\", name)] - rr_all[(\"c3\", \"base\")])\n            print(f\"paired dLB[{lab}] {name} vs base: {d.mean():+.4f} +/- {np.std([d[rng3.integers(0, n, n)].mean() for _ in range(2000)]):.4f}\")\n    # break-even: c3 MRR that pays for the c1/c2 dilution\n    loss = (H.LB_A * (wide.loc[\"base\", \"c1\"] - wide.loc[\"gen blend 0.88\", \"c1\"]) + H.LB_B * (wide.loc[\"base\", \"c2\"] - wide.loc[\"gen blend 0.88\", \"c2\"]))\n    print(f\"gen blend: c1/c2 cost {loss:+.4f} LB; break-even c3 MRR {max(loss, 0) / LB_C:.4f} (measured {wide.loc['gen blend 0.88', 'c3']:.4f})\")\n    sfx = f\"_shuf{a.shuffle}_c3w{a.c3_weight:g}\" + (\"_smoke\" if a.smoke else \"\")\n    pickle.dump({f\"{k[0]}|{k[1]}\": v for k, v in rr_all.items()}, open(cpath(f\"rr_stage2_{a.set}{sfx}.pkl\"), \"wb\"))\n    df.to_csv(cpath(f\"report_stage2_{a.set}{sfx}.csv\"), index=False)\n    log(f\"report -> {cpath(f'report_stage2_{a.set}{sfx}.csv')}\")\n\n\n# ---------------------------------------------------------------------------------------------\n# submit\n# ---------------------------------------------------------------------------------------------\ndef submit(a):\n    \"\"\"The engine on a test file with generated candidates. Rankers are fit once on every row: the shipped files with\n    the np set held out, plus our own np c1/c2/c3 rows with the gen block (the configuration `score` cross-validates).\n    Needs the np caches from `harness.py build` and `stage2.py gen`. --base 1 also writes the same-pipeline submission\n    without generated candidates.\"\"\"\n    import casmi_engine as E, metric, glob, pv_fp\n    import pyarrow.parquet as pq\n    metric.install_dependencies()\n    if a.smoke: E.CFG.GBM = dict(E.CFG.GBM, max_iter=3); E.RANK.GBM = dict(E.RANK.GBM, max_iter=3)\n    P = pickle.load(open(cpath(\"pool.pkl\"), \"rb\")); pool_fp = np.load(cpath(\"pool_fp.npy\"), mmap_mode=\"r\")\n    E.POOL = dict(P, fp=pool_fp)\n    # 1. test channels against the full library (nothing masked)\n    L = E.load_library(H.TRAIN); log(f\"library {len(L['prec']):,} spectra\")\n    L[\"row_p\"] = P[\"k2i\"].reindex(L[\"ik\"]).fillna(-1).values.astype(np.int64)\n    del L[\"ik\"], L[\"smi\"]\n    I = E.build_index(L)\n    single, merged, _ = pv_fp.load_fp_models(sorted(glob.glob(os.path.join(H.DATA, \"fpmodels\", \"fp_*.pt\"))), \"cpu\")\n    te = pq.read_table(a.test).to_pandas()\n    if a.limit:\n        te = te[te.molecule_id.isin(sorted(te.molecule_id.unique())[:a.limit])]\n        samp = pd.read_csv(a.sample); a.sample = cpath(\"sample_limit.csv\")\n        samp[samp.molecule_id.isin(set(te.molecule_id))].to_csv(a.sample, index=False)\n    log(f\"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules\")\n    mols, recs = E.compute_channels(L, I, (single, merged, \"cpu\"), te)\n    del L, I\n    specs = {r[\"mid\"]: list(r[\"specs\"]) for r in recs if \"specs\" in r}  # compute_frag pops them\n    E.compute_frag(recs, a.workers)\n    # 2. generated candidates for the test molecules: every top-80 spectral analog is a source\n    cm = H.canon_map(E, np.concatenate([r[\"cand\"] for r in recs]), a.workers)\n    jobs = [dict(sim=\"test\", mid=r[\"mid\"], target=float(r[\"target\"]),\n                 src=[(P[\"smiles\"][int(p)], float(P[\"mass\"][int(p)]), k, float(s)) for k, (p, s) in enumerate(r[\"ana\"])])\n            for r in recs if len(r[\"cand\"])]\n    res = run_products(jobs, a.workers)\n    Gt = {r[\"mid\"]: finalize(r, res[(\"test\", r[\"mid\"])], specs[r[\"mid\"]], cm, P) for r in recs if (\"test\", r[\"mid\"]) in res}\n    ng = np.array([len(g[\"smiles\"]) for g in Gt.values()])\n    log(f\"test: generated per molecule median {np.median(ng):.0f}, mean {ng.mean():.1f}, max {ng.max()}; \"\n        f\"molecules with any {np.mean(ng > 0):.1%}\")\n    # 3. np ranker sets (c1/c2/c3) and one extended pool over np + test products\n    runs, truth, truth_key, base, G, cm_np = np_sets(E, metric, a)\n    cm.update(cm_np)\n    G[\"test\"] = Gt\n    gidx = extend_pool(E, P, pool_fp, G, cm)\n    ext = {s: extend_recs(base[s], G[s], gidx) for s in base}\n    rng2 = np.random.default_rng(a.shuffle + 1000)\n    for s in (\"c1\", \"c2\", \"c3\"): permute(ext[s], rng2)\n    set_truth(list(base.values()) + list(ext.values()), cm, truth_key)\n    test_ext = extend_recs(recs, Gt, gidx)\n    permute(test_ext, np.random.default_rng(a.shuffle + 2000))  # generated block was appended at the end\n    # 4. final rankers on all rows\n    drop_A, (XA, YA, MA), (XB, YB, SB), blocks = ranker_rows(E, a, base, truth)\n    oA = own_rows3(E, ext, [\"base\", \"gen\"], True)\n    oB = own_rows3(E, ext, blocks + [\"gen\"], False)\n    pad = lambda X: np.hstack([X, np.zeros((len(X), len(GEN_COLS)), np.float32)])\n    t0 = time.time()\n    mA = H.fit_A(E, np.vstack([pad(XA), oA[0]]), np.concatenate([YA, oA[1]]), np.concatenate([MA, oA[2]]),\n                 np.concatenate([np.ones(len(YA)), np.where(oA[2] == 2, a.c3_weight, 1.0)]))\n    mB = H.fit_B(E, np.vstack([pad(XB), oB[0]]), np.concatenate([YB, oB[1]]), np.concatenate([SB, oB[2]]),\n                 np.concatenate([np.ones(len(YB)), np.where(oB[2] == 2, a.c3_weight, 1.0)]))\n    log(f\"gen rankers fit on A {len(YA) + len(oA[1]):,} / B {len(YB) + len(oB[1]):,} rows \"\n        f\"(c3 positives {int(oA[1][oA[2] == 2].sum())}) in {time.time() - t0:.0f}s\")\n    sA = E.score_molecules(test_ext, mA, [\"base\", \"gen\"], True); sB = E.score_molecules(test_ext, mB, blocks + [\"gen\"], False)\n    sc = E.blend_scores(sA, sB, a.w_pv)\n    by = {r[\"mid\"]: r for r in test_ext}\n    top = {m: by[m][\"cand\"][np.argsort(-v, kind=\"mergesort\")] for m, v in sc.items()}\n    log(f\"test: generated candidate ranked 1st for {np.mean([t[0] >= len(P['mass']) for t in top.values()]):.1%} of molecules; \"\n        f\"share of generated in the top 25 {np.mean([(t[:25] >= len(P['mass'])).mean() for t in top.values()]):.1%}\")\n    sub = E.make_submissions(mols, test_ext, {\"gen\": sc}, a.sample, a.workers)[\"gen\"]\n    sub.to_csv(a.out, index=False); log(f\"submission -> {a.out} ({len(sub)} rows)\")\n    if a.base:\n        b2 = {s: {\"recs\": base[s]} for s in (\"c1\", \"c2\")}\n        oA, oB = H.own_rows(E, b2, [\"base\"], True), H.own_rows(E, b2, blocks, False)\n        mA = H.fit_A(E, np.vstack([XA, oA[0]]), np.concatenate([YA, oA[1]]), np.concatenate([MA, oA[2]]), np.ones(len(YA) + len(oA[1])))\n        mB = H.fit_B(E, np.vstack([XB, oB[0]]), np.concatenate([YB, oB[1]]), np.concatenate([SB, oB[2]]), np.ones(len(YB) + len(oB[1])))\n        sc = E.blend_scores(E.score_molecules(recs, mA, [\"base\"], True), E.score_molecules(recs, mB, blocks, False), a.w_pv)\n        out = a.out.replace(\".csv\", \"_base.csv\")\n        E.make_submissions(mols, recs, {\"base\": sc}, a.sample, a.workers)[\"base\"].to_csv(out, index=False)\n        log(f\"same-pipeline baseline -> {out}\")\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"stage\", choices=[\"gen\", \"score\", \"submit\"])\n    ap.add_argument(\"--test\", default=os.path.join(H.DATA, \"comp\", \"test.parquet\"))\n    ap.add_argument(\"--sample\", default=os.path.join(H.DATA, \"comp\", \"sample_submission.csv\"))\n    ap.add_argument(\"--out\", default=\"submission.csv\")\n    ap.add_argument(\"--limit\", type=int, default=0, help=\"submit: first N test molecules only (smoke test)\")\n    ap.add_argument(\"--w_pv\", type=float, default=0.88)\n    ap.add_argument(\"--base\", type=int, default=1, help=\"submit: also write the same-pipeline no-generation submission\")\n    ap.add_argument(\"--set\", default=\"np\")\n    ap.add_argument(\"--folds\", type=int, default=5)\n    ap.add_argument(\"--shuffle\", type=int, default=1)\n    ap.add_argument(\"--c3_weight\", type=float, default=1.0, help=\"sample-weight multiplier on c3 rows in the gen rankers\")\n    ap.add_argument(\"--workers\", type=int, default=10)\n    ap.add_argument(\"--smoke\", type=int, default=0, help=\"tiny GBMs, to test the whole path\")\n    a = ap.parse_args()\n    {\"gen\": gen, \"score\": score, \"submit\": submit}[a.stage](a)\n"}
import os, sys, glob, json, subprocess, time
W = "/kaggle/working/casmi"; ENG = f"{W}/engine"; DATA = f"{W}/data"
os.makedirs(ENG, exist_ok=True); os.makedirs(f"{DATA}/cache", exist_ok=True)
for name, code in SRC.items():
    open(f"{ENG}/{name}", "w").write(code)

def one(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True), key=len)
    assert hits, pattern
    return hits[0]

# RDKit pinned to the scorer's version (TautomerEnumerator output changes between releases)
try:
    import rdkit
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q", one("rdkit-2026.3.3-*.whl")], check=True)
    import rdkit
print("rdkit", rdkit.__version__); assert rdkit.__version__ == "2026.03.3"

# the harness' local data layout, as symlinks
LAYOUT = {"comp": ["train.parquet", "test.parquet", "sample_submission.csv"], "coconut": ["coco_fp.npy", "coco_mass.npy", "coco_meta.pkl", "fp_bits.npy"],
          "fpmodels": ["fp_merged_m1.pt", "fp_single_s2.pt"], "pvrank": ["rank_train.npz"], "megayak": ["sim_rank_rows_nofp.npz"]}
for d, names in LAYOUT.items():
    os.makedirs(f"{DATA}/{d}", exist_ok=True)
    for n in names:
        dst = f"{DATA}/{d}/{n}"
        if not os.path.exists(dst): os.symlink(one(n), dst)
        print(dst, "->", os.path.realpath(dst))
NCPU = os.cpu_count(); print("cpus", NCPU)

def run(*args):
    """Run an engine stage, stream its output, keep a log next to the cache."""
    t0 = time.time(); log = open(f"{DATA}/{args[0].replace('.py', '')}_{args[1]}.log", "a")
    p = subprocess.Popen([sys.executable, "-u", *args, "--workers", str(NCPU)], cwd=ENG, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=dict(os.environ, CASMI_ROOTS=DATA, PYTHONWARNINGS="ignore"))
    for line in p.stdout:
        if "Warning" in line: continue
        print(line, end=""); log.write(line); log.flush()
    assert p.wait() == 0, f"{args} exited {p.returncode}"
    print(f"== {args} done in {(time.time() - t0) / 60:.1f} min")


## 2 · Build retrieval features and the simulated candidate sets

This stage masks the relevant training rows, builds spectral channels and fingerprints, and scores the C1/C2/C3 simulation sets. In the saved run, the build processed **2,539,608 library spectra** and formed a pool of **712,199 structures**. It took **64.4 minutes** on the recorded Kaggle runtime.

**Insight.** The pool is large relative to the 250 simulated molecules. Retrieval and candidate ranking therefore do most of the filtering; the later generated block is an additional source of candidates rather than a replacement for the library.


In [ ]:
run("harness.py", "build", "--set", "np")

## 3 · Generate one-step NP candidates

Each simulated molecule is paired with its top-80 spectral analogs as transformation sources. Generated products are canonicalized and deduplicated against the candidate window before they are added to the ranking pool.

| Simulation split | Generated candidates per molecule (mean) | Truth among generated |
|---|---:|---:|
| C1 | 10.4 | 0.0% |
| C2 | 10.0 | 0.0% |
| C3 | 10.3 | 38.8% |

**Insight.** C3 demonstrates that the transformation rules can recover the known answer for a meaningful fraction of these simulated cases. C1/C2 show no truth among generated candidates in this run, so C3 coverage should be read as split-specific evidence, not a general recovery rate. Generation and caching took **11.3 minutes**.


In [ ]:
run("stage2.py", "gen", "--set", "np")

## 4 · Fit the rankers and write the submission

The final stage extends the pool with generated products, refits Ranker A and Ranker B, blends their scores at 0.88, and writes both the generated-candidate submission and a no-generation baseline.

**Test-set diagnostics from the saved run**

- Generated products were available for **44.8%** of molecules; the median count was **0**, the mean **3.3**, and the maximum **33**.
- A generated product ranked first for **0.8%** of molecules.
- Generated products occupied **11.2%** of the top-25 positions on average.
- The output contains **400 rows**. This run stage took **50.1 minutes**.

**Insight.** The generated block has sparse test-set coverage, but it does enter the top-25 ranking when present. The saved diagnostics do not establish whether it improves the competition metric; compare `submission.csv` with `submission_base.csv` using the official evaluation metric.


In [ ]:
run("stage2.py", "submit", "--set", "np", "--c3_weight", "1", "--out", "/kaggle/working/submission.csv")

## 5 · Export diagnostics and submission artifacts

This cell copies logs and stage-2 reports into `/kaggle/working/out`, displays a preview of the submission, then removes the temporary working tree to keep the Kaggle output small.

### Artifacts

- `submission.csv` — blended ranking with generated candidates.
- `submission_base.csv` — same pipeline without generated candidates.
- `/kaggle/working/out/` — available logs and simulation reports.

The notebook's saved output confirms a **400 × 2** submission table with `molecule_id` and `smiles` columns.


In [ ]:
import pandas as pd, shutil
os.makedirs("/kaggle/working/out", exist_ok=True)
for f in glob.glob(f"{DATA}/*.log") + glob.glob(f"{DATA}/cache/report_stage2_*.csv") + glob.glob(f"{DATA}/cache/rr_stage2_*.pkl") \
        + glob.glob(f"{DATA}/cache/gen_stats_*.csv"):
    shutil.copy(f, "/kaggle/working/out/")
if glob.glob("/kaggle/working/out/report_stage2_*.csv"):
    print(pd.read_csv(glob.glob("/kaggle/working/out/report_stage2_*.csv")[0]).round(4).to_string(index=False))
if os.path.exists("/kaggle/working/submission.csv"):
    sub = pd.read_csv("/kaggle/working/submission.csv"); print(sub.shape); print(sub.head())
shutil.rmtree(W)  # keep the kernel output small (pool fingerprints, fitted rankers)